# Machine Learning Project 2

## 1. Questions

### 1.1 "The main goal of this model [Linear Regression] is to estimate the linear function of the training set with the least error"

$$ MSE = \sum_{i=1}^{N} (y_{n} - f(x_{n}; θ)^{2} $$

Given that f(x;θ)=wTx+b+ϵ=i=1∑D​wi​xi​+b+ϵ,

$$ MSE = \sum_{i=1}^{N} (y_{i} - w^{T}x_{i})^{2}) $$


X = training data vector, y = target vector, w= weights vector

Vector form of the equation:
$$ L(w) = 1/N ||y - Xw||^{2} = 1/N (y - Xw)^{T}(y - Xw) = 1/N [y^{T}y - y^{T}Xw - (Xw)^{T}y + (Xw)^{T}(Xw)] $$

$$ (Xw)^{T}y = w^{T}X^{T}y = (y^{T}Xw)^{T} = y^{T}Xw => $$

$$ => y^{T}y - y^{T}Xw - (Xw)^{T}y + (Xw)^{T}(Xw) = y^{T}y - y^{T}Xw - y^{T}Xw + (Xw)^{T}(Xw) = y^{T}y - 2y^{T}Xw + (Xw)^{T}(Xw) $$

$$ (Xw)^{T}(Xw) = w^{T}X^{T}Xw => $$

$$ L(w) = 1/N [y^{T}y - 2y^{T}Xw + w^{T}X^{T}Xw] $$

Derivative: $$ L(w)' = dL/dw = 1/N(-2X^{T}y + 2X^{T}Xw) = 0 $$
$$ X^{T}y + X^{T}Xw = 0  \qquad(*(X^{T}X)^{-1})$$
$$ -X^{T}y = X^{T}Xw  \qquad(*(X^{T}X)^{-1})$$
$$ X^{T}Xw * (X^{T}X)^{-1} = X^{T}y * (X^{T}X)^{-1} $$
$$ w = X^{T}y * (X^{T}X)^{-1} $$

### 1.2 What changes in the solution when L1 and L2 regularizations are added to the loss function.

Ridge, vector form:
$$ L(w) = 1/N ||y - Xw||^{2} + lambda||w||^{2} $$

Lasso, vector form:
$$ L(w) = 1/N ||y - Xw||^{2} + lambda||w|| $$

### 1.3 Why L1 regularization is often used to select features. Why are there many weights equal to 0 after the model is fit?

Answer: this happens due to the nature of L1's gradient. It's constant (lambda only). While for the Ridge model, it's 2*w\*lambda. So the penalty remains regardless of the weight's value.

## 2. Introduction

In [1]:
# Libraries
!pip install pandas numpy scikit-learn lightgbm scipy statsmodels matplotlib seaborn


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np

In [3]:
# Train.json
df_train = pd.read_json('data/train.json')
print(df_train.head())

    bathrooms  bedrooms                       building_id  \
4         1.0         1  8579a0b0d54db803821a35a4a615e97a   
6         1.0         2  b8e75fc949a6cd8225b455648a951712   
9         1.0         2  cd759a988b8f23924b5a2058d5ab2b49   
10        1.5         3  53a5b119ba8f7b61d4e010512e0dfc85   
15        1.0         0  bfb9405149bfff42a92980b594c28234   

                created                                        description  \
4   2016-06-16 05:55:27  Spacious 1 Bedroom 1 Bathroom in Williamsburg!...   
6   2016-06-01 05:44:33  BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...   
9   2016-06-14 15:19:59  **FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...   
10  2016-06-24 07:54:24  A Brand New 3 Bedroom 1.5 bath ApartmentEnjoy ...   
15  2016-06-28 03:50:23  Over-sized Studio w abundant closets. Availabl...   

        display_address                                           features  \
4   145 Borinquen Place  [Dining Room, Pre-War, Laundry in Building, Di...   
6       

In [4]:
# Test.json
df_test = pd.read_json('data/test.json')
print(df_test.head())

   bathrooms  bedrooms                       building_id              created  \
0        1.0         1  79780be1514f645d7e6be99a3de696c5  2016-06-11 05:29:41   
1        1.0         2                                 0  2016-06-24 06:36:34   
2        1.0         0                                 0  2016-06-17 01:23:39   
3        1.0         2  f9c826104b91d868e69bd25746448c0c  2016-06-21 05:06:02   
5        1.0         1  81062936e12ee5fa6cd2b965698e17d5  2016-06-16 07:24:27   

                                         description  display_address  \
0  Large with awesome terrace--accessible via bed...   Suffolk Street   
1  Prime Soho - between Bleecker and Houston - Ne...  Thompson Street   
2  Spacious studio in Prime Location. Cleanbuildi...  Sullivan Street   
3  For immediate access call Bryan.<br /><br />Bo...     Jones Street   
5  Beautiful TRUE 1 bedroom in a luxury building ...   Exchange Place   

                                            features  latitude  listing_id

In [5]:
# Size of data
print("Data shape")
print(f"Train.json. Rows, columns: \n{df_train.shape}")
print(f"Test.json. Rows, columns: \n{df_test.shape}")

Data shape
Train.json. Rows, columns: 
(49352, 15)
Test.json. Rows, columns: 
(74659, 14)


In [6]:
# Lists of columns
print("Lists of columns")
print(f"\nTrain.json: \n{list(df_train.columns)}")
print(f"\nTest.json: \n{list(df_test.columns)}")

Lists of columns

Train.json: 
['bathrooms', 'bedrooms', 'building_id', 'created', 'description', 'display_address', 'features', 'latitude', 'listing_id', 'longitude', 'manager_id', 'photos', 'price', 'street_address', 'interest_level']

Test.json: 
['bathrooms', 'bedrooms', 'building_id', 'created', 'description', 'display_address', 'features', 'latitude', 'listing_id', 'longitude', 'manager_id', 'photos', 'price', 'street_address']


### Target column: 'price'

In [9]:
# Quick analysis of the data: the info(), describe(), corr() methods
print("Basic data analysis")
print("\nData set name: train.json")
print(df_train.info())
print()
print(df_train.describe())
print()
print(df_train.corr(numeric_only=True))
print("\nData set name: test.json")
print(df_test.info())
print()
print(df_test.describe())
print()
print(df_test.corr(numeric_only=True))

Basic data analysis

Data set name: train.json
<class 'pandas.core.frame.DataFrame'>
Index: 49352 entries, 4 to 124009
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   bathrooms        49352 non-null  float64
 1   bedrooms         49352 non-null  int64  
 2   building_id      49352 non-null  object 
 3   created          49352 non-null  object 
 4   description      49352 non-null  object 
 5   display_address  49352 non-null  object 
 6   features         49352 non-null  object 
 7   latitude         49352 non-null  float64
 8   listing_id       49352 non-null  int64  
 9   longitude        49352 non-null  float64
 10  manager_id       49352 non-null  object 
 11  photos           49352 non-null  object 
 12  price            49352 non-null  int64  
 13  street_address   49352 non-null  object 
 14  interest_level   49352 non-null  object 
dtypes: float64(3), int64(3), object(9)
memory usage: 6.0+ MB
None

In [10]:
# Check for empty columns
empty_cols_train = df_train.isnull().all().sum()
empty_cols_test = df_test.isnull().all().sum()

print(f"Number of empty columns, train.json: {empty_cols_train}")
print(f"Number of empty columns, test.json: {empty_cols_test}")

Number of empty columns, train.json: 0
Number of empty columns, test.json: 0


In [11]:
# Deleting rows outside the 1st and 99th percentiles
print("REMOVING OUTLIERS")
# Train
print("\nTrain.json")
p1_train = df_train['price'].quantile(0.01)
p99_train = df_train['price'].quantile(0.99)

print(f"1st percentile: USD {p1_train:,.0f}")
print(f"99th percentile: USD {p99_train:,.0f}")

cleaned_df_train = df_train[(df_train['price'] >= p1_train) & (df_train['price'] <= p99_train)]

print(f"\nOriginal train dataset length: {len(df_train):,} apartments")
print(f"After removing 1st and 99th: {len(cleaned_df_train):,} apartments") 
print(f"Removed: {len(df_train) - len(cleaned_df_train):,} apts, which correspond to {((len(df_train) - len(cleaned_df_train))/len(df_train)*100):.1f}% of the original dataset")

#------------------------

# Test
print("\nTest.json")
p1_test = df_test['price'].quantile(0.01)
p99_test = df_test['price'].quantile(0.99)

print(f"1st percentile: USD {p1_test:,.0f}")
print(f"99th percentile: USD {p99_test:,.0f}")

cleaned_df_test = df_test[(df_test['price'] >= p1_test) & (df_test['price'] <= p99_test)]

print(f"\nOriginal test dataset length: {len(df_test):,} apartments")
print(f"After removing 1st and 99th: {len(cleaned_df_test):,} apartments") 
print(f"Removed: {len(df_test) - len(cleaned_df_test):,} apts, which correspond to {((len(df_test) - len(cleaned_df_test))/len(df_test)*100):.1f}% of the original dataset")

REMOVING OUTLIERS

Train.json
1st percentile: USD 1,475
99th percentile: USD 13,000

Original train dataset length: 49,352 apartments
After removing 1st and 99th: 48,379 apartments
Removed: 973 apts, which correspond to 2.0% of the original dataset

Test.json
1st percentile: USD 1,495
99th percentile: USD 13,000

Original test dataset length: 74,659 apartments
After removing 1st and 99th: 73,216 apartments
Removed: 1,443 apts, which correspond to 1.9% of the original dataset


### Interest level processing

In [12]:
print("INTEREST LEVEL DATA TYPE")
print(f"Train.json: {df_train['interest_level'].dtype}")
if 'interest_level' in df_test:
    print(f"Test.json: {df_test['interest_level'].dtype}")
else:
    print("Test.json: column 'interest_level' not found in the test.json data set")

INTEREST LEVEL DATA TYPE
Train.json: object
Test.json: column 'interest_level' not found in the test.json data set


**Comment:** As there was no 'interest_level' column found in the test.json file, the further processing encompasses the train.json data set only.

In [13]:
print("COLUMN 'interest_level' VALUES")
print("Cleaned train data set")
print(f"\nNumber of unique values: {cleaned_df_train['interest_level'].nunique()}")
print("VALUES-values:")
print(cleaned_df_train['interest_level'].value_counts().index.tolist())

print("\nVALUE COUNTS")
print(cleaned_df_train['interest_level'].value_counts().head())

COLUMN 'interest_level' VALUES
Cleaned train data set

Number of unique values: 3
VALUES-values:
['low', 'medium', 'high']

VALUE COUNTS
interest_level
low       33697
medium    11116
high       3566
Name: count, dtype: int64


In [14]:
# Encoding
cleaned_df_train = cleaned_df_train.copy()

interest_mapping = {
    'low': 0,
    'medium': 1, 
    'high': 2
}

cleaned_df_train['interest_level_encoded'] = cleaned_df_train['interest_level'].map(interest_mapping)

print("VALUES ENCODED")
interest_lvl_table = cleaned_df_train[['interest_level', 'interest_level_encoded']].value_counts().sort_index()
print(interest_lvl_table)

VALUES ENCODED
interest_level  interest_level_encoded
high            2                          3566
low             0                         33697
medium          1                         11116
Name: count, dtype: int64


## 3. Intro Data analysis part 2

In [15]:
features_column = cleaned_df_train['features']
print(features_column)

4         [Dining Room, Pre-War, Laundry in Building, Di...
6         [Doorman, Elevator, Laundry in Building, Dishw...
9         [Doorman, Elevator, Laundry in Building, Laund...
10                                                       []
15        [Doorman, Elevator, Fitness Center, Laundry in...
                                ...                        
124000              [Elevator, Dishwasher, Hardwood Floors]
124002    [Common Outdoor Space, Cats Allowed, Dogs Allo...
124004    [Dining Room, Elevator, Pre-War, Laundry in Bu...
124008    [Pre-War, Laundry in Unit, Dishwasher, No Fee,...
124009    [Dining Room, Elevator, Laundry in Building, D...
Name: features, Length: 48379, dtype: object


In [17]:
# Function for removing [,], ', ", and space from the 'Features' column
def clean_features(features_list):
    if not isinstance(features_list, list):
        return []

    cleaned = []
    for feature in features_list:
        if isinstance(feature, str):
            cleaned_feature = feature.replace('[', '').replace(']', '').replace("'", '').replace('"', '').replace(' ', '')
            if cleaned_feature:
                cleaned.append(cleaned_feature)
    return cleaned
print("Status: symbols [, ], ', \", and space removed")

# Function application
cleaned_df_train['features'] = cleaned_df_train['features'].apply(clean_features)
cleaned_df_test['features'] = cleaned_df_test['features'].apply(clean_features)
print(cleaned_df_train['features'].head())

Status: symbols [, ], ', ", and space removed
4     [DiningRoom, Pre-War, LaundryinBuilding, Dishw...
6     [Doorman, Elevator, LaundryinBuilding, Dishwas...
9     [Doorman, Elevator, LaundryinBuilding, Laundry...
10                                                   []
15    [Doorman, Elevator, FitnessCenter, LaundryinBu...
Name: features, dtype: object


/var/folders/wx/__g_4vf96gv181qvbr6551g40000gn/T/ipykernel_1083/1447449718.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_df_test['features'] = cleaned_df_test['features'].apply(clean_features)


In [18]:
print("Sample features:")
print(cleaned_df_train['features'].head())
print(f"Type of first feature: {type(cleaned_df_train['features'].iloc[0])}")
print(f"\nSample feature value: {cleaned_df_train['features'].iloc[0]}")

Sample features:
4     [DiningRoom, Pre-War, LaundryinBuilding, Dishw...
6     [Doorman, Elevator, LaundryinBuilding, Dishwas...
9     [Doorman, Elevator, LaundryinBuilding, Laundry...
10                                                   []
15    [Doorman, Elevator, FitnessCenter, LaundryinBu...
Name: features, dtype: object
Type of first feature: <class 'list'>

Sample feature value: ['DiningRoom', 'Pre-War', 'LaundryinBuilding', 'Dishwasher', 'HardwoodFloors', 'DogsAllowed', 'CatsAllowed']


In [19]:
def extract_all_features(features_series):
    all_features = []
    for feature_list in features_series:
        if isinstance(feature_list, list) and feature_list: ##only process if it's a non-empty list
            all_features.extend(feature_list)
    return all_features

all_features = extract_all_features(cleaned_df_train['features'])
print(f"Total features extracted: {len(all_features)}")
print(f"Sample features: {all_features[:10]}")

# Top 20
from collections import Counter
feature_counter = Counter(all_features)
top_20_features = [feature for feature, count in feature_counter.most_common(20)]
print(f"\nTop 20 features: {top_20_features}")

Total features extracted: 262573
Sample features: ['DiningRoom', 'Pre-War', 'LaundryinBuilding', 'Dishwasher', 'HardwoodFloors', 'DogsAllowed', 'CatsAllowed', 'Doorman', 'Elevator', 'LaundryinBuilding']

Top 20 features: ['Elevator', 'HardwoodFloors', 'CatsAllowed', 'DogsAllowed', 'Doorman', 'Dishwasher', 'NoFee', 'LaundryinBuilding', 'FitnessCenter', 'Pre-War', 'LaundryinUnit', 'RoofDeck', 'OutdoorSpace', 'DiningRoom', 'HighSpeedInternet', 'Balcony', 'SwimmingPool', 'LaundryInBuilding', 'NewConstruction', 'Terrace']


In [20]:
# Binaries for the top's
for feature in top_20_features:
    cleaned_df_train[feature] = cleaned_df_train['features'].apply(lambda x: 1 if feature in x else 0)
    cleaned_df_test[feature] = cleaned_df_test['features'].apply(lambda x: 1 if feature in x else 0)

# New feature list
feature_list = top_20_features + ['bathrooms', 'bedrooms']

print(f"Number of features created: {len(feature_list)}")
print(feature_list)

/var/folders/wx/__g_4vf96gv181qvbr6551g40000gn/T/ipykernel_1083/942633525.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_df_test[feature] = cleaned_df_test['features'].apply(lambda x: 1 if feature in x else 0)
/var/folders/wx/__g_4vf96gv181qvbr6551g40000gn/T/ipykernel_1083/942633525.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_df_test[feature] = cleaned_df_test['features'].apply(lambda x: 1 if feature in x else 0)
/var/folders/wx/__g_4vf96gv181qvbr6551g40000gn/T/ipykernel_1083/94

Number of features created: 22
['Elevator', 'HardwoodFloors', 'CatsAllowed', 'DogsAllowed', 'Doorman', 'Dishwasher', 'NoFee', 'LaundryinBuilding', 'FitnessCenter', 'Pre-War', 'LaundryinUnit', 'RoofDeck', 'OutdoorSpace', 'DiningRoom', 'HighSpeedInternet', 'Balcony', 'SwimmingPool', 'LaundryInBuilding', 'NewConstruction', 'Terrace', 'bathrooms', 'bedrooms']


/var/folders/wx/__g_4vf96gv181qvbr6551g40000gn/T/ipykernel_1083/942633525.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_df_test[feature] = cleaned_df_test['features'].apply(lambda x: 1 if feature in x else 0)
/var/folders/wx/__g_4vf96gv181qvbr6551g40000gn/T/ipykernel_1083/942633525.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_df_test[feature] = cleaned_df_test['features'].apply(lambda x: 1 if feature in x else 0)
/var/folders/wx/__g_4vf96gv181qvbr6551g40000gn/T/ipykernel_1083/94

In [25]:
# To fix the warnings above, cpies are made:
print("Status: copying data sets to avoid the warnings (see the cell above)")

cleaned_df_test = cleaned_df_test.copy()
cleaned_df_train = cleaned_df_train.copy()

# Feature cleaning for the copied data sets:
print("Status: applying feature cleaning to the test data set")
cleaned_df_test['features'] = cleaned_df_test['features'].apply(clean_features)

# Creating binary features for test data set without warnings
print("Status: creating binary features for the test data set")
for feature in top_20_features:
    cleaned_df_test[feature] = cleaned_df_test['features'].apply(lambda x: 1 if feature in x else 0)

print("Status: preprocessing completed without warnings")

# Verifying if data is ready
print(f"\nDATA VERIFICATION")
print(f"Train data set shape: {cleaned_df_train.shape}")
print(f"Test data set shape: {cleaned_df_test.shape}")

# Checking features
missing_train = [f for f in feature_list if f not in cleaned_df_train.columns]
missing_test = [f for f in feature_list if f not in cleaned_df_test.columns]

if not missing_train and not missing_test:
    print("\nVerivication results: all features exist in both train and test data sets")
    # Creating training arrays
    X_train = cleaned_df_train[feature_list].values
    y_train = cleaned_df_train['price'].values
    X_test = cleaned_df_test[feature_list].values
    y_test = cleaned_df_test['price'].values
    print(f"X_train shape: {X_train.shape}")
    print(f"X_test shape: {X_test.shape}")
else:
    print(f"Missing in train: {missing_train}")
    print(f"Missing in test: {missing_test}")

Status: copying data sets to avoid the warnings (see the cell above)
Status: applying feature cleaning to the test data set
Status: creating binary features for the test data set
Status: preprocessing completed without warnings

DATA VERIFICATION
Train data set shape: (48379, 36)
Test data set shape: (73216, 34)

Verivication results: all features exist in both train and test data sets
X_train shape: (48379, 22)
X_test shape: (73216, 22)


## 4. Models implementation - Linear regression

In [26]:
# Libraries
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd

In [28]:
np.random.seed(21)

class LinearRegressionSGD:
    def __init__(self, learning_rate=0.005, n_iterations=800, random_state=21):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.random_state = random_state

    def fit(self, X, y):
        np.random.seed(self.random_state)
        X = np.column_stack([np.ones(X.shape[0]), X])
        self.weights = np.random.normal(0, 0.01, X.shape[1])

        for i in range(self.n_iterations):
            indices = np.arange(X.shape[0])
            np.random.shuffle(indices)

            for idx in indices:
                xi = X[idx:idx+1]
                yi = y[idx:idx+1]

                # Prediction
                y_pred = xi.dot(self.weights)
                # Gradient calculation
                gradient = -2 * xi.T.dot(yi - y_pred) / 1  ## divided by 1 for single sample
                # Updating weights
                self.weights -= self.learning_rate * gradient.flatten()
    def predict(self, X):
        # Bias:
        X = np.column_stack([np.ones(X.shape[0]), X])
        return X.dot(self.weights)

# 4.4 R2 function
def r2_coefficient(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)  ## sum of squares of residuals
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2) ## total sum of squares
    return 1 - (ss_res / ss_tot)

In [35]:
# 4.5 Making predictions with the algorithm
## preparing data for training
X_train = cleaned_df_train[feature_list].values
y_train = cleaned_df_train['price'].values
X_test = cleaned_df_test[feature_list].values
y_test = cleaned_df_test['price'].values

print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")

############################

##training linear regression
print("\nStatus: training custom Linear Regression with SGD")
lr_custom = LinearRegressionSGD(learning_rate=0.005, n_iterations=800)
lr_custom.fit(X_train, y_train)

# Predictions
y_train_pred_custom = lr_custom.predict(X_train)
y_test_pred_custom = lr_custom.predict(X_test)

# Metrics for custom implementation. MAE, RMSE, R2
train_mae_custom = mean_absolute_error(y_train, y_train_pred_custom)
train_rmse_custom = np.sqrt(mean_squared_error(y_train, y_train_pred_custom))
train_r2_custom = r2_coefficient(y_train, y_train_pred_custom)

test_mae_custom = mean_absolute_error(y_test, y_test_pred_custom)
test_rmse_custom = np.sqrt(mean_squared_error(y_test, y_test_pred_custom))
test_r2_custom = r2_coefficient(y_test, y_test_pred_custom)

print(f"CUSTOM LINEAR REGRESSION RESULTS")
print("Data set: Train")
print(f"MAE {train_mae_custom:.4f} \\ RMSE {train_rmse_custom:.4f} \\ R2 {train_r2_custom:.4f}")
print("Data set: Test")
print(f"MAE {test_mae_custom:.4f} \\ RMSE {test_rmse_custom:.4f} \\ R2 {test_r2_custom:.4f}")

# 4.6 Training sklearn linear regression
print("\nStatus: training sklearn Linear Regression")
lr_sklearn = LinearRegression()
lr_sklearn.fit(X_train, y_train)

# Predictions
y_train_pred_sklearn = lr_sklearn.predict(X_train)
y_test_pred_sklearn = lr_sklearn.predict(X_test)

# Metrics for sklearn implementation. MAE, RMSE, R2
train_mae_sklearn = mean_absolute_error(y_train, y_train_pred_sklearn)
train_rmse_sklearn = np.sqrt(mean_squared_error(y_train, y_train_pred_sklearn))
train_r2_sklearn = r2_score(y_train, y_train_pred_sklearn)

test_mae_sklearn = mean_absolute_error(y_test, y_test_pred_sklearn)
test_rmse_sklearn = np.sqrt(mean_squared_error(y_test, y_test_pred_sklearn))
test_r2_sklearn = r2_score(y_test, y_test_pred_sklearn)

print(f"SKLEARN LINEAR REGRESSION RESULTS")
print("Data set: Train")
print(f"MAE {train_mae_sklearn:.4f} \\ RMSE {train_rmse_sklearn:.4f} \\ R2 {train_r2_sklearn:.4f}")
print("Data set: Test")
print(f"MAE {test_mae_sklearn:.4f} \\ RMSE {test_rmse_sklearn:.4f} \\ R2 {test_r2_sklearn:.4f}")

Training data shape: (48379, 22)
Test data shape: (73216, 22)

Status: training custom Linear Regression with SGD
CUSTOM LINEAR REGRESSION RESULTS
Data set: Train
MAE 714.4845 \ RMSE 1041.2831 \ R2 0.5752
Data set: Test
MAE 718.5126 \ RMSE 1204.3433 \ R2 0.4263

Status: training sklearn Linear Regression
SKLEARN LINEAR REGRESSION RESULTS
Data set: Train
MAE 711.7912 \ RMSE 1035.3516 \ R2 0.5800
Data set: Test
MAE 716.8458 \ RMSE 1226.8654 \ R2 0.4047


In [36]:
# 4.7 Comparing implementations
print(f"IMPLEMENTATION COMPARISON (CUSTOM minus SKLEARN")
print(f"MAE delta (train): {abs(train_mae_custom - train_mae_sklearn):.4f}")
print(f"MAE delta (test): {abs(test_mae_custom - test_mae_sklearn):.4f}")
print(f"RMSE delta (train): {abs(train_rmse_custom - train_rmse_sklearn):.4f}")
print(f"RMSE delta (test): {abs(test_rmse_custom - test_rmse_sklearn):.4f}")
print(f"R2 delta (train): {abs(train_r2_custom - train_r2_sklearn):.4f}")
print(f"R2 delta (test): {abs(test_r2_custom - test_r2_sklearn):.4f}")

# 4.8 Table for the metrics
results_mae = pd.DataFrame({
    'model': ['LinearRegression_Custom', 'LinearRegression_Sklearn'],
    'train': [train_mae_custom, train_mae_sklearn],
    'test': [test_mae_custom, test_mae_sklearn]
})

results_rmse = pd.DataFrame({
    'model': ['LinearRegression_Custom', 'LinearRegression_Sklearn'],
    'train': [train_rmse_custom, train_rmse_sklearn],
    'test': [test_rmse_custom, test_rmse_sklearn]
})

results_r2 = pd.DataFrame({
    'model': ['LinearRegression_Custom', 'LinearRegression_Sklearn'],
    'train': [train_r2_custom, train_r2_sklearn],
    'test': [test_r2_custom, test_r2_sklearn]
})

print("\nMAE results table:")
print(results_mae)
print("\nRMSE results table:")
print(results_rmse)
print("\nR2 results table:")
print(results_r2)

IMPLEMENTATION COMPARISON (CUSTOM minus SKLEARN
MAE delta (train): 2.6934
MAE delta (test): 1.6668
RMSE delta (train): 5.9315
RMSE delta (test): 22.5221
R2 delta (train): 0.0048
R2 delta (test): 0.0217

MAE results table:
                      model       train        test
0   LinearRegression_Custom  714.484532  718.512561
1  LinearRegression_Sklearn  711.791166  716.845780

RMSE results table:
                      model        train         test
0   LinearRegression_Custom  1041.283065  1204.343345
1  LinearRegression_Sklearn  1035.351576  1226.865399

R2 results table:
                      model     train      test
0   LinearRegression_Custom  0.575208  0.426336
1  LinearRegression_Sklearn  0.580034  0.404680


## 5. Regularized models implementation — Ridge, Lasso, and ElasticNet 

In [37]:
import numpy as np
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd

np.random.seed(21)

# 5.1 Ridge Regression (L2)
class RidgeRegressionSGD:
    def __init__(self, learning_rate=0.005, n_iterations=800, alpha=1.0, random_state=21):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.alpha = alpha  ## L2 regularization parameter
        self.random_state = random_state
        
    def fit(self, X, y):
        np.random.seed(self.random_state)
        X = np.column_stack([np.ones(X.shape[0]), X])
        self.weights = np.random.normal(0, 0.01, X.shape[1])
        
        for i in range(self.n_iterations):
            indices = np.arange(X.shape[0])
            np.random.shuffle(indices)
            
            for idx in indices:
                xi = X[idx:idx+1]
                yi = y[idx:idx+1]
                
                # Prediction
                y_pred = xi.dot(self.weights)
                
                # Gradient calculation with L2 regularization
                mse_gradient = -2 * xi.T.dot(yi - y_pred)
                l2_gradient = np.zeros_like(self.weights)
                l2_gradient[1:] = 2 * self.alpha * self.weights[1:]  ## skipping bias term
                
                gradient = mse_gradient.flatten() + l2_gradient
                self.weights -= self.learning_rate * gradient
    
    def predict(self, X):
        X = np.column_stack([np.ones(X.shape[0]), X])
        return X.dot(self.weights)

# 5.1 Lasso Regression (L1)
class LassoRegressionSGD:
    def __init__(self, learning_rate=0.005, n_iterations=800, alpha=1.0, random_state=21):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.alpha = alpha  ## L1 regularization parameter
        self.random_state = random_state
        
    def fit(self, X, y):
        np.random.seed(self.random_state)
        X = np.column_stack([np.ones(X.shape[0]), X])
        self.weights = np.random.normal(0, 0.01, X.shape[1])
        
        for i in range(self.n_iterations):
            indices = np.arange(X.shape[0])
            np.random.shuffle(indices)
            
            for idx in indices:
                xi = X[idx:idx+1]
                yi = y[idx:idx+1]
                
                # Prediction
                y_pred = xi.dot(self.weights)
                
                # Gradient calculation with L1 regularization
                mse_gradient = -2 * xi.T.dot(yi - y_pred)
                l1_gradient = np.zeros_like(self.weights)
                l1_gradient[1:] = self.alpha * np.sign(self.weights[1:])  ## skipping bias term
                
                gradient = mse_gradient.flatten() + l1_gradient
                self.weights -= self.learning_rate * gradient
    
    def predict(self, X):
        X = np.column_stack([np.ones(X.shape[0]), X])
        return X.dot(self.weights)

# 5.1 ElasticNet Regression (L1 + L2 reg)
class ElasticNetRegressionSGD:
    def __init__(self, learning_rate=0.005, n_iterations=800, alpha=1.0, l1_ratio=0.5, random_state=21):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.alpha = alpha  ## overall regularization strength
        self.l1_ratio = l1_ratio  ## balance between L1 and L2 (0=Ridge, 1=Lasso)
        self.random_state = random_state
        
    def fit(self, X, y):
        np.random.seed(self.random_state)
        X = np.column_stack([np.ones(X.shape[0]), X])
        self.weights = np.random.normal(0, 0.01, X.shape[1])
        
        for i in range(self.n_iterations):
            indices = np.arange(X.shape[0])
            np.random.shuffle(indices)
            
            for idx in indices:
                xi = X[idx:idx+1]
                yi = y[idx:idx+1]
                
                # Prediction
                y_pred = xi.dot(self.weights)
                
                # Gradient calculation with L1 + L2 regularization
                # MSE gradient: -2 * xi.T.dot(yi - y_pred)
                # L1 component: alpha * l1_ratio * sign(weights)
                # L2 component: alpha * (1 - l1_ratio) * 2 * weights
                mse_gradient = -2 * xi.T.dot(yi - y_pred)
                
                reg_gradient = np.zeros_like(self.weights)
                # Skipping bias term (index 0)
                l1_component = self.alpha * self.l1_ratio * np.sign(self.weights[1:])
                l2_component = self.alpha * (1 - self.l1_ratio) * 2 * self.weights[1:]
                reg_gradient[1:] = l1_component + l2_component
                
                gradient = mse_gradient.flatten() + reg_gradient
                self.weights -= self.learning_rate * gradient
    
    def predict(self, X):
        X = np.column_stack([np.ones(X.shape[0]), X])
        return X.dot(self.weights)

print("Status: custom regularized models implemented")
print("Ridge == L2 regularization (alpha * sum(weights^2))")
print("Lasso == L1 regularization (alpha * sum(|weights|))")
print("ElasticNet == L1 + L2 regularization")

Status: custom regularized models implemented
Ridge == L2 regularization (alpha * sum(weights^2))
Lasso == L1 regularization (alpha * sum(|weights|))
ElasticNet == L1 + L2 regularization


In [38]:
# 5.2 Makung predictions with the algorithm
## Training custom models
print("TRAINING CUSTOM REGULARIZED MODELS")

# Ridge
print("\nStatus: training custom Ridge regression")
ridge_custom = RidgeRegressionSGD(learning_rate=0.005, n_iterations=800, alpha=1.0)
ridge_custom.fit(X_train, y_train)

y_train_pred_ridge_custom = ridge_custom.predict(X_train)
y_test_pred_ridge_custom = ridge_custom.predict(X_test)

# Metrics
train_mae_ridge_custom = mean_absolute_error(y_train, y_train_pred_ridge_custom)
train_rmse_ridge_custom = np.sqrt(mean_squared_error(y_train, y_train_pred_ridge_custom))
train_r2_ridge_custom = r2_coefficient(y_train, y_train_pred_ridge_custom)

test_mae_ridge_custom = mean_absolute_error(y_test, y_test_pred_ridge_custom)
test_rmse_ridge_custom = np.sqrt(mean_squared_error(y_test, y_test_pred_ridge_custom))
test_r2_ridge_custom = r2_coefficient(y_test, y_test_pred_ridge_custom)

print(f"CUSTOM RIDGE RESULTS")
print("Dataset: train")
print(f"MAE {train_mae_ridge_custom:.4f} \\ RMSE {train_rmse_ridge_custom:.4f} \\ R2 {train_r2_ridge_custom:.4f}")
print("Dataset: test")
print(f"MAE: {test_mae_ridge_custom:.4f} \\ RMSE {test_rmse_ridge_custom:.4f} \\ R2 {test_r2_ridge_custom:.4f}")

# Lasso
print("\nStatus:Training custom Lasso regression")
lasso_custom = LassoRegressionSGD(learning_rate=0.005, n_iterations=800, alpha=0.1)
lasso_custom.fit(X_train, y_train)

y_train_pred_lasso_custom = lasso_custom.predict(X_train)
y_test_pred_lasso_custom = lasso_custom.predict(X_test)

#Metrics
train_mae_lasso_custom = mean_absolute_error(y_train, y_train_pred_lasso_custom)
train_rmse_lasso_custom = np.sqrt(mean_squared_error(y_train, y_train_pred_lasso_custom))
train_r2_lasso_custom = r2_coefficient(y_train, y_train_pred_lasso_custom)

test_mae_lasso_custom = mean_absolute_error(y_test, y_test_pred_lasso_custom)
test_rmse_lasso_custom = np.sqrt(mean_squared_error(y_test, y_test_pred_lasso_custom))
test_r2_lasso_custom = r2_coefficient(y_test, y_test_pred_lasso_custom)

print(f"CUSTOM LASSO RESULTS")
print("Dataset: train")
print(f"MAE {train_mae_lasso_custom:.4f} \\ RMSE {train_rmse_lasso_custom:.4f} \\ R2 {train_r2_lasso_custom:.4f}")
print("Dataset: test")
print(f"MAE {test_mae_lasso_custom:.4f} \\ RMSE {test_rmse_lasso_custom:.4f} \\ R2 {test_r2_lasso_custom:.4f}")

# ElasticNet
print("\nStatus: training custom ElasticNet regression")
elastic_custom = ElasticNetRegressionSGD(learning_rate=0.005, n_iterations=800, alpha=0.1, l1_ratio=0.5)
elastic_custom.fit(X_train, y_train)

y_train_pred_elastic_custom = elastic_custom.predict(X_train)
y_test_pred_elastic_custom = elastic_custom.predict(X_test)

# Metrics
train_mae_elastic_custom = mean_absolute_error(y_train, y_train_pred_elastic_custom)
train_rmse_elastic_custom = np.sqrt(mean_squared_error(y_train, y_train_pred_elastic_custom))
train_r2_elastic_custom = r2_coefficient(y_train, y_train_pred_elastic_custom)

test_mae_elastic_custom = mean_absolute_error(y_test, y_test_pred_elastic_custom)
test_rmse_elastic_custom = np.sqrt(mean_squared_error(y_test, y_test_pred_elastic_custom))
test_r2_elastic_custom = r2_coefficient(y_test, y_test_pred_elastic_custom)

print(f"CUSTOM ELASTICNET RESULTS")
print("Dataset: train")
print(f"MAE {train_mae_elastic_custom:.4f} \\ RMSE {train_rmse_elastic_custom:.4f} \\ R2 {train_r2_elastic_custom:.4f}")
print("Dataset: test")
print(f"MAE {test_mae_elastic_custom:.4f} \\ RMSE {test_rmse_elastic_custom:.4f} \\ R2 {test_r2_elastic_custom:.4f}")

TRAINING CUSTOM REGULARIZED MODELS

Status: training custom Ridge regression
CUSTOM RIDGE RESULTS
Dataset: train
MAE 899.6467 \ RMSE 1280.7597 \ R2 0.3574
Dataset: test
MAE: 901.9485 \ RMSE 1284.5636 \ R2 0.3474

Status:Training custom Lasso regression
CUSTOM LASSO RESULTS
Dataset: train
MAE 714.4725 \ RMSE 1041.2817 \ R2 0.5752
Dataset: test
MAE 718.5009 \ RMSE 1204.3267 \ R2 0.4264

Status: training custom ElasticNet regression
CUSTOM ELASTICNET RESULTS
Dataset: train
MAE 724.3653 \ RMSE 1060.5533 \ R2 0.5593
Dataset: test
MAE 727.7288 \ RMSE 1158.2320 \ R2 0.4694


In [39]:
# 5.3 Sklearn Ridge, Lasso, and ElasticNet models
## Training sklearn Ridge, Lasso, and ElasticNet models
print("TRAINING SKLEARN REGULARIZED MODELS")

# Ridge
print("Status: Training sklearn Ridge regression")
ridge_sklearn = Ridge(alpha=1.0, random_state=21)
ridge_sklearn.fit(X_train, y_train)

y_train_pred_ridge_sklearn = ridge_sklearn.predict(X_train)
y_test_pred_ridge_sklearn = ridge_sklearn.predict(X_test)

# Metrics
train_mae_ridge_sklearn = mean_absolute_error(y_train, y_train_pred_ridge_sklearn)
train_rmse_ridge_sklearn = np.sqrt(mean_squared_error(y_train, y_train_pred_ridge_sklearn))
train_r2_ridge_sklearn = r2_score(y_train, y_train_pred_ridge_sklearn)

test_mae_ridge_sklearn = mean_absolute_error(y_test, y_test_pred_ridge_sklearn)
test_rmse_ridge_sklearn = np.sqrt(mean_squared_error(y_test, y_test_pred_ridge_sklearn))
test_r2_ridge_sklearn = r2_score(y_test, y_test_pred_ridge_sklearn)

print(f"SKLEARN RIDGE RESULTS")
print("Dataset: train")
print(f"MAE {train_mae_ridge_sklearn:.4f} \\ RMSE {train_rmse_ridge_sklearn:.4f} \\ R2 {train_r2_ridge_sklearn:.4f}")
print("Dataset: test")
print(f"MAE {test_mae_ridge_sklearn:.4f} \\ RMSE {test_rmse_ridge_sklearn:.4f} \\ R2 {test_r2_ridge_sklearn:.4f}")

# Lasso
print("\nStatus: training sklearn Lasso regression")
lasso_sklearn = Lasso(alpha=0.1, random_state=21)
lasso_sklearn.fit(X_train, y_train)

y_train_pred_lasso_sklearn = lasso_sklearn.predict(X_train)
y_test_pred_lasso_sklearn = lasso_sklearn.predict(X_test)

# Metrics
train_mae_lasso_sklearn = mean_absolute_error(y_train, y_train_pred_lasso_sklearn)
train_rmse_lasso_sklearn = np.sqrt(mean_squared_error(y_train, y_train_pred_lasso_sklearn))
train_r2_lasso_sklearn = r2_score(y_train, y_train_pred_lasso_sklearn)

test_mae_lasso_sklearn = mean_absolute_error(y_test, y_test_pred_lasso_sklearn)
test_rmse_lasso_sklearn = np.sqrt(mean_squared_error(y_test, y_test_pred_lasso_sklearn))
test_r2_lasso_sklearn = r2_score(y_test, y_test_pred_lasso_sklearn)

print(f"SKLEARN LASSO RESULTS")
print("Data set: train")
print(f"MAE {train_mae_lasso_sklearn:.4f} \\ RMSE {train_rmse_lasso_sklearn:.4f} \\ R2 {train_r2_lasso_sklearn:.4f}")
print("Data set: test")
print(f"MAE {test_mae_lasso_sklearn:.4f} \\ RMSE {test_rmse_lasso_sklearn:.4f} \\ R2 {test_r2_lasso_sklearn:.4f}")

# ElasticNet
print("\nStatus: training sklearn ElasticNet regression")
elastic_sklearn = ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=21)
elastic_sklearn.fit(X_train, y_train)

y_train_pred_elastic_sklearn = elastic_sklearn.predict(X_train)
y_test_pred_elastic_sklearn = elastic_sklearn.predict(X_test)

# Metrics
train_mae_elastic_sklearn = mean_absolute_error(y_train, y_train_pred_elastic_sklearn)
train_rmse_elastic_sklearn = np.sqrt(mean_squared_error(y_train, y_train_pred_elastic_sklearn))
train_r2_elastic_sklearn = r2_score(y_train, y_train_pred_elastic_sklearn)

test_mae_elastic_sklearn = mean_absolute_error(y_test, y_test_pred_elastic_sklearn)
test_rmse_elastic_sklearn = np.sqrt(mean_squared_error(y_test, y_test_pred_elastic_sklearn))
test_r2_elastic_sklearn = r2_score(y_test, y_test_pred_elastic_sklearn)

print(f"SKLEARN ELASTICNET RESULTS")
print("Data set: train")
print(f"MAE {train_mae_elastic_sklearn:.4f} \\ RMSE {train_rmse_elastic_sklearn:.4f} \\ R2 {train_r2_elastic_sklearn:.4f}")
print("Data set: test")
print(f"MAE {test_mae_elastic_sklearn:.4f} \\ RMSE {test_rmse_elastic_sklearn:.4f} \\ R2 {test_r2_elastic_sklearn:.4f}")

TRAINING SKLEARN REGULARIZED MODELS
Status: Training sklearn Ridge regression
SKLEARN RIDGE RESULTS
Dataset: train
MAE 711.7880 \ RMSE 1035.3516 \ R2 0.5800
Dataset: test
MAE 716.8419 \ RMSE 1226.8232 \ R2 0.4047

Status: training sklearn Lasso regression
SKLEARN LASSO RESULTS
Data set: train
MAE 711.7304 \ RMSE 1035.3535 \ R2 0.5800
Data set: test
MAE 716.7811 \ RMSE 1226.8346 \ R2 0.4047

Status: training sklearn ElasticNet regression
SKLEARN ELASTICNET RESULTS
Data set: train
MAE 716.5861 \ RMSE 1049.6254 \ R2 0.5684
Data set: test
MAE 720.5893 \ RMSE 1168.4990 \ R2 0.4600


In [41]:
# 5.4 Comparing custom and sklearn implementations
print("Comparing custom and sklearn implementations (delta == custom minus sklearn)")

print(f"\nRIDGE DELTAS")
print(f"MAE train: {abs(train_mae_ridge_custom - train_mae_ridge_sklearn):.4f}")
print(f"MAE test: {abs(test_mae_ridge_custom - test_mae_ridge_sklearn):.4f}")
print(f"RMSE train: {abs(train_rmse_ridge_custom - train_rmse_ridge_sklearn):.4f}")
print(f"RMSE test: {abs(test_rmse_ridge_custom - test_rmse_ridge_sklearn):.4f}")
print(f"R2 train: {abs(train_r2_ridge_custom - train_r2_ridge_sklearn):.4f}")
print(f"R2 test: {abs(test_r2_ridge_custom - test_r2_ridge_sklearn):.4f}")

print(f"\nLASSO DELTAS")
print(f"MAE train: {abs(train_mae_lasso_custom - train_mae_lasso_sklearn):.4f}")
print(f"MAE test: {abs(test_mae_lasso_custom - test_mae_lasso_sklearn):.4f}")
print(f"RMSE train: {abs(train_rmse_lasso_custom - train_rmse_lasso_sklearn):.4f}")
print(f"RMSE test: {abs(test_rmse_lasso_custom - test_rmse_lasso_sklearn):.4f}")
print(f"R2 train: {abs(train_r2_lasso_custom - train_r2_lasso_sklearn):.4f}")
print(f"R2 test: {abs(test_r2_lasso_custom - test_r2_lasso_sklearn):.4f}")

print(f"\nELASTICNET DELTAS")
print(f"MAE train: {abs(train_mae_elastic_custom - train_mae_elastic_sklearn):.4f}")
print(f"MAE test: {abs(test_mae_elastic_custom - test_mae_elastic_sklearn):.4f}")
print(f"RMSE train: {abs(train_rmse_elastic_custom - train_rmse_elastic_sklearn):.4f}")
print(f"RMSE test: {abs(test_rmse_elastic_custom - test_rmse_elastic_sklearn):.4f}")
print(f"R2 train: {abs(train_r2_elastic_custom - train_r2_elastic_sklearn):.4f}")
print(f"R2 test: {abs(test_r2_elastic_custom - test_r2_elastic_sklearn):.4f}")

# 5.5 Updating results tables
print("\n___UPDATED RESULTS TABLES___")

# Updating MAE results
new_mae_rows = [
    ['Ridge_Custom', train_mae_ridge_custom, test_mae_ridge_custom],
    ['Ridge_Sklearn', train_mae_ridge_sklearn, test_mae_ridge_sklearn],
    ['Lasso_Custom', train_mae_lasso_custom, test_mae_lasso_custom],
    ['Lasso_Sklearn', train_mae_lasso_sklearn, test_mae_lasso_sklearn],
    ['ElasticNet_Custom', train_mae_elastic_custom, test_mae_elastic_custom],
    ['ElasticNet_Sklearn', train_mae_elastic_sklearn, test_mae_elastic_sklearn]
]

for row in new_mae_rows:
    results_mae.loc[len(results_mae)] = row

# Updating RMSE results
new_rmse_rows = [
    ['Ridge_Custom', train_rmse_ridge_custom, test_rmse_ridge_custom],
    ['Ridge_Sklearn', train_rmse_ridge_sklearn, test_rmse_ridge_sklearn],
    ['Lasso_Custom', train_rmse_lasso_custom, test_rmse_lasso_custom],
    ['Lasso_Sklearn', train_rmse_lasso_sklearn, test_rmse_lasso_sklearn],
    ['ElasticNet_Custom', train_rmse_elastic_custom, test_rmse_elastic_custom],
    ['ElasticNet_Sklearn', train_rmse_elastic_sklearn, test_rmse_elastic_sklearn]
]

for row in new_rmse_rows:
    results_rmse.loc[len(results_rmse)] = row

# Updating R2 results
new_r2_rows = [
    ['Ridge_Custom', train_r2_ridge_custom, test_r2_ridge_custom],
    ['Ridge_Sklearn', train_r2_ridge_sklearn, test_r2_ridge_sklearn],
    ['Lasso_Custom', train_r2_lasso_custom, test_r2_lasso_custom],
    ['Lasso_Sklearn', train_r2_lasso_sklearn, test_r2_lasso_sklearn],
    ['ElasticNet_Custom', train_r2_elastic_custom, test_r2_elastic_custom],
    ['ElasticNet_Sklearn', train_r2_elastic_sklearn, test_r2_elastic_sklearn]
]

for row in new_r2_rows:
    results_r2.loc[len(results_r2)] = row

print("\nUpdated MAE results table:")
print(results_mae)
print("\nUpdated RMSE results table:")
print(results_rmse)
print("\nUpdated R2 results table:")
print(results_r2)

Comparing custom and sklearn implementations (delta == custom minus sklearn)

RIDGE DELTAS
MAE train: 187.8587
MAE test: 185.1067
RMSE train: 245.4081
RMSE test: 57.7404
R2 train: 0.2227
R2 test: 0.0574

LASSO DELTAS
MAE train: 2.7421
MAE test: 1.7198
RMSE train: 5.9282
RMSE test: 22.5079
R2 train: 0.0048
R2 test: 0.0216

ELASTICNET DELTAS
MAE train: 7.7792
MAE test: 7.1395
RMSE train: 10.9279
RMSE test: 10.2670
R2 train: 0.0090
R2 test: 0.0094

___UPDATED RESULTS TABLES___

Updated MAE results table:
                       model       train        test
0    LinearRegression_Custom  714.484532  718.512561
1   LinearRegression_Sklearn  711.791166  716.845780
2               Ridge_Custom  899.646690  901.948541
3              Ridge_Sklearn  711.787958  716.841873
4               Lasso_Custom  714.472530  718.500932
5              Lasso_Sklearn  711.730426  716.781148
6          ElasticNet_Custom  724.365273  727.728814
7         ElasticNet_Sklearn  716.586109  720.589267
8               

## 6. Feature normalization

In [43]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import pandas as pd

# 6.1 Normalization examples
print("MANDATORY normalization:")
print("Distance-based algorithms (KNN, K-means): larger scale features dominate distance calculations, so normalization prevents them from data misrepresentation.")
print("Neural networks: again, prevents data from being misrepresented, plus makes sure there's no gradient explosion due to scale differences.")
print("Principal Component Analysis (PCA): variance-based, dominated by high-scale features. Without normalization, the latters might distort the results.")

print("\nOPTIONAL normalization:")
print("Random Forest: splits are scale-invariant, what's important is the order of feature values to find split points.")
print("Naive Bayes: it works with conditional probabilities where the scale of the features doesn't affect the relative probabilities or the final classification outcome.")
print("Decision trees: they use relative ordering and are not affected by the absolute value or scale of features.")

# 6.2 MinMaxScaler mathematical formula
print("\nMinMaxScaler mathematical formula")
print("Formula: X_normalized = (X - X_min) / (X_max - X_min)")
print("X_min = minimum value in the feature")
print("X_max = maximum value in the feature")
print("All the features get scaled to range [0, 1]")

MANDATORY normalization:
Distance-based algorithms (KNN, K-means): larger scale features dominate distance calculations, so normalization prevents them from data misrepresentation.
Neural networks: again, prevents data from being misrepresented, plus makes sure there's no gradient explosion due to scale differences.
Principal Component Analysis (PCA): variance-based, dominated by high-scale features. Without normalization, the latters might distort the results.

OPTIONAL normalization:
Random Forest: splits are scale-invariant, what's important is the order of feature values to find split points.
Naive Bayes: it works with conditional probabilities where the scale of the features doesn't affect the relative probabilities or the final classification outcome.
Decision trees: they use relative ordering and are not affected by the absolute value or scale of features.

MinMaxScaler mathematical formula
Formula: X_normalized = (X - X_min) / (X_max - X_min)
X_min = minimum value in the featur

In [44]:
# 6.3 Custom MinMaxScaler
class CustomMinMaxScaler:
    def __init__(self):
        self.min_ = None
        self.max_ = None
        self.scale_ = None
        
    def fit(self, X):
        X = np.array(X)
        self.min_ = np.min(X, axis=0)
        self.max_ = np.max(X, axis=0)
        self.scale_ = self.max_ - self.min_
        # Handling constant features (min == max)
        self.scale_[self.scale_ == 0] = 1.0
        return self
    
    def transform(self, X):
        if self.min_ is None or self.max_ is None:
            raise ValueError("Must fit scaler before transform")
        X = np.array(X)
        return (X - self.min_) / self.scale_
    
    def fit_transform(self, X):
        return self.fit(X).transform(X)
    
    def inverse_transform(self, X):
        if self.min_ is None or self.max_ is None:
            raise ValueError("Must fit scaler before inverse_transform")
        X = np.array(X)
        return X * self.scale_ + self.min_

# 6.4 Sklearn MinMaxScaler
# Test on a sample of the data for comparison
sample_X = X_train[:1000]  # first 1000 samples for quick comparison

# Custom MinMaxScaler
custom_minmax = CustomMinMaxScaler()
X_custom_minmax = custom_minmax.fit_transform(sample_X)

# Sklearn MinMaxScaler  
sklearn_minmax = MinMaxScaler()
X_sklearn_minmax = sklearn_minmax.fit_transform(sample_X)

# 6.5 Custom vs. sklearn results comparison
difference_minmax = np.abs(X_custom_minmax - X_sklearn_minmax)
max_difference_minmax = np.max(difference_minmax)
mean_difference_minmax = np.mean(difference_minmax)

print(f"MinMaxScaler comparison on {sample_X.shape[0]} samples:")
print(f"Maximum difference: {max_difference_minmax:.10f}")
print(f"Mean difference: {mean_difference_minmax:.10f}")
print(f"Custom min values: {custom_minmax.min_[:3]}")  # Show first 3
print(f"Sklearn min values: {sklearn_minmax.data_min_[:3]}")
print(f"Custom max values: {custom_minmax.max_[:3]}")
print(f"Sklearn max values: {sklearn_minmax.data_max_[:3]}")

if max_difference_minmax < 1e-10:    ## if the biggest difference is less than 1e-10, consider numbers equal
    print("Result: Custom MinMaxScaler matches sklearn implementation")
else:
    print("Result: Implementations don't match")

MinMaxScaler comparison on 1000 samples:
Maximum difference: 0.0000000000
Mean difference: 0.0000000000
Custom min values: [0. 0. 0.]
Sklearn min values: [0. 0. 0.]
Custom max values: [1. 1. 1.]
Sklearn max values: [1. 1. 1.]
Result: Custom MinMaxScaler matches sklearn implementation


In [46]:
# 6.6 StandardScaler formula and implementation
print("StandardScaler mathematical formula")
print("Formula: Z = (X - μ) / σ")
print("Z = the standardized value (the Z-score)")
print("X = the individual feature value")
print("μ = the mean of the feature")
print("σ = standard deviation of the feature")
print("All features have mean=0, std=1 (standard normal distribution)")

# Implementation
class CustomStandardScaler:
    def __init__(self):
        self.mean_ = None
        self.std_ = None
        
    def fit(self, X):
        X = np.array(X)
        self.mean_ = np.mean(X, axis=0)
        self.std_ = np.std(X, axis=0, ddof=0)  ## population std, not sample std
        # Handle constant features, where std == 0
        self.std_[self.std_ == 0] = 1.0
        return self
    
    def transform(self, X):
        if self.mean_ is None or self.std_ is None:
            raise ValueError("Must fit scaler before transform")
        X = np.array(X)
        return (X - self.mean_) / self.std_  ##scaling features to have mean=0, std=1"""
    
    def fit_transform(self, X):
        return self.fit(X).transform(X)
    
    def inverse_transform(self, X):
        if self.mean_ is None or self.std_ is None:
            raise ValueError("Must fit scaler before inverse_transform")
        X = np.array(X)
        return X * self.std_ + self.mean_

# Custom StandardScaler implementation
custom_standard = CustomStandardScaler()
X_custom_standard = custom_standard.fit_transform(sample_X)

# Sklearn StandardScaler
sklearn_standard = StandardScaler()
X_sklearn_standard = sklearn_standard.fit_transform(sample_X)

# Results comparison
difference_standard = np.abs(X_custom_standard - X_sklearn_standard)
max_difference_standard = np.max(difference_standard)
mean_difference_standard = np.mean(difference_standard)

print(f"\nStandardScaler comparison on {sample_X.shape[0]} samples:")
print(f"Maximum difference: {max_difference_standard:.10f}")
print(f"Mean difference: {mean_difference_standard:.10f}")
print(f"Custom mean values: {custom_standard.mean_[:3]}")
print(f"Sklearn mean values: {sklearn_standard.mean_[:3]}")
print(f"Custom std values: {custom_standard.std_[:3]}")
print(f"Sklearn std values: {sklearn_standard.scale_[:3]}")  # sklearn calls it scale_

if max_difference_standard < 1e-10:    ## if the biggest difference is less than 1e-10, consider numbers equal
    print("Result: Custom StandardScaler matches sklearn implementation")
else:
    print("Result: Implementations don't match")

print(f"\nStatus: both normalization methods implemented and verified")

StandardScaler mathematical formula
Formula: Z = (X - μ) / σ
Z = the standardized value (the Z-score)
X = the individual feature value
μ = the mean of the feature
σ = standard deviation of the feature
All features have mean=0, std=1 (standard normal distribution)

StandardScaler comparison on 1000 samples:
Maximum difference: 0.0000000000
Mean difference: 0.0000000000
Custom mean values: [0.497 0.483 0.487]
Sklearn mean values: [0.497 0.483 0.487]
Custom std values: [0.499991   0.49971092 0.49983097]
Sklearn std values: [0.499991   0.49971092 0.49983097]
Result: Custom StandardScaler matches sklearn implementation

Status: both normalization methods implemented and verified


In [49]:
# Preparing normalized datasets for full training
print("PREPARING NORMALIZED DATASETS FOR MODEL TRAINING")

# MinMaxScaler normalization
print("\nStatus: Applying MinMaxScaler to the full dataset")
minmax_scaler = MinMaxScaler()
X_train_minmax = minmax_scaler.fit_transform(X_train)
X_test_minmax = minmax_scaler.transform(X_test)

print(f"MinMax normalized train shape: {X_train_minmax.shape}")
print(f"MinMax normalized test shape: {X_test_minmax.shape}")
print(f"MinMax train range: [{X_train_minmax.min():.3f}, {X_train_minmax.max():.3f}]")
print(f"MinMax test range: [{X_test_minmax.min():.3f}, {X_test_minmax.max():.3f}]")

# StandardScaler normalization  
print("\nStatus: Applying StandardScaler to the full dataset")
standard_scaler = StandardScaler()
X_train_standard = standard_scaler.fit_transform(X_train)
X_test_standard = standard_scaler.transform(X_test)

print(f"Standard normalized train shape: {X_train_standard.shape}")
print(f"Standard normalized test shape: {X_test_standard.shape}")
print(f"Standard train mean: {X_train_standard.mean():.6f} (should be ~0)")
print(f"Standard train std: {X_train_standard.std():.6f} (should be ~1)")
print(f"Standard test mean: {X_test_standard.mean():.6f}")
print(f"Standard test std: {X_test_standard.std():.6f}")

print("\nDatasets available:")
print("X_train, X_test (original features)")
print("X_train_minmax, X_test_minmax (MinMax normalized)")
print("X_train_standard, X_test_standard (Standard normalized)")
print("y_train, y_test (targets; no normalization needed)")

PREPARING NORMALIZED DATASETS FOR MODEL TRAINING

Status: Applying MinMaxScaler to the full dataset
MinMax normalized train shape: (48379, 22)
MinMax normalized test shape: (73216, 22)
MinMax train range: [0.000, 1.000]
MinMax test range: [0.000, 11.200]

Status: Applying StandardScaler to the full dataset
Standard normalized train shape: (48379, 22)
Standard normalized test shape: (73216, 22)
Standard train mean: -0.000000 (should be ~0)
Standard train std: 1.000000 (should be ~1)
Standard test mean: -0.000346
Standard test std: 1.018931

Datasets available:
X_train, X_test (original features)
X_train_minmax, X_test_minmax (MinMax normalized)
X_train_standard, X_test_standard (Standard normalized)
y_train, y_test (targets; no normalization needed)


## 7. Custom and sklearn models with normalized data

In [50]:
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd

# Using same hyperparameters as before
learning_rate = 0.005
n_iterations = 800

# 7.1 Fiting models with MinMaxScaler
print("TRAINING MODELS WITH MINMAX NORMALIZATION")

# MinMax Linear Regression
print("\nStatus: Training Linear Regression (MinMax)")
lr_minmax_custom = LinearRegressionSGD(learning_rate=learning_rate, n_iterations=n_iterations)
lr_minmax_custom.fit(X_train_minmax, y_train)

y_train_pred_lr_minmax = lr_minmax_custom.predict(X_train_minmax)
y_test_pred_lr_minmax = lr_minmax_custom.predict(X_test_minmax)

train_mae_lr_minmax = mean_absolute_error(y_train, y_train_pred_lr_minmax)
train_rmse_lr_minmax = np.sqrt(mean_squared_error(y_train, y_train_pred_lr_minmax))
train_r2_lr_minmax = r2_coefficient(y_train, y_train_pred_lr_minmax)

test_mae_lr_minmax = mean_absolute_error(y_test, y_test_pred_lr_minmax)
test_rmse_lr_minmax = np.sqrt(mean_squared_error(y_test, y_test_pred_lr_minmax))
test_r2_lr_minmax = r2_coefficient(y_test, y_test_pred_lr_minmax)

print("Dataset: Train")
print(f"MAE {train_mae_lr_minmax:.4f} \\ RMSE {train_rmse_lr_minmax:.4f} \\ R2 {train_r2_lr_minmax:.4f}")
print("Dataset: Test")
print(f"MAE {test_mae_lr_minmax:.4f} \\ RMSE {test_rmse_lr_minmax:.4f} \\ R2 {test_r2_lr_minmax:.4f}")

# MinMax Ridge
print("\nStatus: Training Ridge (MinMax)")
ridge_minmax_custom = RidgeRegressionSGD(learning_rate=learning_rate, n_iterations=n_iterations, alpha=1.0)
ridge_minmax_custom.fit(X_train_minmax, y_train)

y_train_pred_ridge_minmax = ridge_minmax_custom.predict(X_train_minmax)
y_test_pred_ridge_minmax = ridge_minmax_custom.predict(X_test_minmax)

train_mae_ridge_minmax = mean_absolute_error(y_train, y_train_pred_ridge_minmax)
train_rmse_ridge_minmax = np.sqrt(mean_squared_error(y_train, y_train_pred_ridge_minmax))
train_r2_ridge_minmax = r2_coefficient(y_train, y_train_pred_ridge_minmax)

test_mae_ridge_minmax = mean_absolute_error(y_test, y_test_pred_ridge_minmax)
test_rmse_ridge_minmax = np.sqrt(mean_squared_error(y_test, y_test_pred_ridge_minmax))
test_r2_ridge_minmax = r2_coefficient(y_test, y_test_pred_ridge_minmax)

print("Dataset: Train")
print(f"MAE {train_mae_ridge_minmax:.4f} \\ RMSE {train_rmse_ridge_minmax:.4f} \\ R2{train_r2_ridge_minmax:.4f}")
print("Dataset: Test")
print(f"MAE {test_mae_ridge_minmax:.4f} \\ RMSE {test_rmse_ridge_minmax:.4f} \\ R2 {test_r2_ridge_minmax:.4f}")

# MinMax Lasso
print("\nStatus: Training Lasso (MinMax)")
lasso_minmax_custom = LassoRegressionSGD(learning_rate=learning_rate, n_iterations=n_iterations, alpha=0.1)
lasso_minmax_custom.fit(X_train_minmax, y_train)

y_train_pred_lasso_minmax = lasso_minmax_custom.predict(X_train_minmax)
y_test_pred_lasso_minmax = lasso_minmax_custom.predict(X_test_minmax)

train_mae_lasso_minmax = mean_absolute_error(y_train, y_train_pred_lasso_minmax)
train_rmse_lasso_minmax = np.sqrt(mean_squared_error(y_train, y_train_pred_lasso_minmax))
train_r2_lasso_minmax = r2_coefficient(y_train, y_train_pred_lasso_minmax)

test_mae_lasso_minmax = mean_absolute_error(y_test, y_test_pred_lasso_minmax)
test_rmse_lasso_minmax = np.sqrt(mean_squared_error(y_test, y_test_pred_lasso_minmax))
test_r2_lasso_minmax = r2_coefficient(y_test, y_test_pred_lasso_minmax)

print("Dataset: Train")
print(f"MAE={train_mae_lasso_minmax:.4f} \\ RMSE {train_rmse_lasso_minmax:.4f} \\ R2 {train_r2_lasso_minmax:.4f}")
print("Dataset: Test")
print(f"MAE {test_mae_lasso_minmax:.4f} \\ RMSE {test_rmse_lasso_minmax:.4f} \\ R2 {test_r2_lasso_minmax:.4f}")

# MinMax ElasticNet
print("\nStatus: Training ElasticNet (MinMax)")
elastic_minmax_custom = ElasticNetRegressionSGD(learning_rate=learning_rate, n_iterations=n_iterations, alpha=0.1, l1_ratio=0.5)
elastic_minmax_custom.fit(X_train_minmax, y_train)

y_train_pred_elastic_minmax = elastic_minmax_custom.predict(X_train_minmax)
y_test_pred_elastic_minmax = elastic_minmax_custom.predict(X_test_minmax)

train_mae_elastic_minmax = mean_absolute_error(y_train, y_train_pred_elastic_minmax)
train_rmse_elastic_minmax = np.sqrt(mean_squared_error(y_train, y_train_pred_elastic_minmax))
train_r2_elastic_minmax = r2_coefficient(y_train, y_train_pred_elastic_minmax)

test_mae_elastic_minmax = mean_absolute_error(y_test, y_test_pred_elastic_minmax)
test_rmse_elastic_minmax = np.sqrt(mean_squared_error(y_test, y_test_pred_elastic_minmax))
test_r2_elastic_minmax = r2_coefficient(y_test, y_test_pred_elastic_minmax)

print("Dataset: Train")
print(f"MAE {train_mae_elastic_minmax:.4f} \\ RMSE {train_rmse_elastic_minmax:.4f} \\ R2 {train_r2_elastic_minmax:.4f}")
print("Dataset: Test")
print(f"MAE {test_mae_elastic_minmax:.4f} \\ RMSE {test_rmse_elastic_minmax:.4f} \\ R2 {test_r2_elastic_minmax:.4f}")

TRAINING MODELS WITH MINMAX NORMALIZATION

Status: Training Linear Regression (MinMax)
Dataset: Train
MAE 710.3205 \ RMSE 1041.4760 \ R2 0.5751
Dataset: Test
MAE 714.7003 \ RMSE 1230.1064 \ R2 0.4015

Status: Training Ridge (MinMax)
Dataset: Train
MAE 1074.0953 \ RMSE 1540.1948 \ R20.0706
Dataset: Test
MAE 1072.6657 \ RMSE 1532.3099 \ R2 0.0714

Status: Training Lasso (MinMax)
Dataset: Train
MAE=710.2863 \ RMSE 1041.4754 \ R2 0.5751
Dataset: Test
MAE 714.6543 \ RMSE 1229.4913 \ R2 0.4021

Status: Training ElasticNet (MinMax)
Dataset: Train
MAE 914.4424 \ RMSE 1350.4864 \ R2 0.2855
Dataset: Test
MAE 913.7223 \ RMSE 1343.3921 \ R2 0.2862


In [51]:
# 7.2 Fiting models with StandardScaler
print("TRAINING MODELS WITH STANDARD NORMALIZATION")

# Standard Linear Regression
print("\nStatus: Training Linear Regression (standard)")
lr_standard_custom = LinearRegressionSGD(learning_rate=learning_rate, n_iterations=n_iterations)
lr_standard_custom.fit(X_train_standard, y_train)

y_train_pred_lr_standard = lr_standard_custom.predict(X_train_standard)
y_test_pred_lr_standard = lr_standard_custom.predict(X_test_standard)

train_mae_lr_standard = mean_absolute_error(y_train, y_train_pred_lr_standard)
train_rmse_lr_standard = np.sqrt(mean_squared_error(y_train, y_train_pred_lr_standard))
train_r2_lr_standard = r2_coefficient(y_train, y_train_pred_lr_standard)

test_mae_lr_standard = mean_absolute_error(y_test, y_test_pred_lr_standard)
test_rmse_lr_standard = np.sqrt(mean_squared_error(y_test, y_test_pred_lr_standard))
test_r2_lr_standard = r2_coefficient(y_test, y_test_pred_lr_standard)

print("Dataset: Train")
print(f"MAE {train_mae_lr_standard:.4f} \\ RMSE {train_rmse_lr_standard:.4f} \\ R2 {train_r2_lr_standard:.4f}")
print("Dataset: Test")
print(f"MAE {test_mae_lr_standard:.4f} \\ RMSE {test_rmse_lr_standard:.4f} \\ R2 {test_r2_lr_standard:.4f}")

# Standard Ridge
print("\nStatus: Training Ridge (standard)")
ridge_standard_custom = RidgeRegressionSGD(learning_rate=learning_rate, n_iterations=n_iterations, alpha=1.0)
ridge_standard_custom.fit(X_train_standard, y_train)

y_train_pred_ridge_standard = ridge_standard_custom.predict(X_train_standard)
y_test_pred_ridge_standard = ridge_standard_custom.predict(X_test_standard)

train_mae_ridge_standard = mean_absolute_error(y_train, y_train_pred_ridge_standard)
train_rmse_ridge_standard = np.sqrt(mean_squared_error(y_train, y_train_pred_ridge_standard))
train_r2_ridge_standard = r2_coefficient(y_train, y_train_pred_ridge_standard)

test_mae_ridge_standard = mean_absolute_error(y_test, y_test_pred_ridge_standard)
test_rmse_ridge_standard = np.sqrt(mean_squared_error(y_test, y_test_pred_ridge_standard))
test_r2_ridge_standard = r2_coefficient(y_test, y_test_pred_ridge_standard)

print("Dataset: Train")
print(f"MAE {train_mae_ridge_standard:.4f} \\ RMSE {train_rmse_ridge_standard:.4f} \\ R2 {train_r2_ridge_standard:.4f}")
print("Dataset: Test")
print(f"MAE {test_mae_ridge_standard:.4f} \\ RMSE {test_rmse_ridge_standard:.4f} \\ R2 {test_r2_ridge_standard:.4f}")

# Standard Lasso
print("\nStatus: Training Lasso (standard)")
lasso_standard_custom = LassoRegressionSGD(learning_rate=learning_rate, n_iterations=n_iterations, alpha=0.1)
lasso_standard_custom.fit(X_train_standard, y_train)

y_train_pred_lasso_standard = lasso_standard_custom.predict(X_train_standard)
y_test_pred_lasso_standard = lasso_standard_custom.predict(X_test_standard)

train_mae_lasso_standard = mean_absolute_error(y_train, y_train_pred_lasso_standard)
train_rmse_lasso_standard = np.sqrt(mean_squared_error(y_train, y_train_pred_lasso_standard))
train_r2_lasso_standard = r2_coefficient(y_train, y_train_pred_lasso_standard)

test_mae_lasso_standard = mean_absolute_error(y_test, y_test_pred_lasso_standard)
test_rmse_lasso_standard = np.sqrt(mean_squared_error(y_test, y_test_pred_lasso_standard))
test_r2_lasso_standard = r2_coefficient(y_test, y_test_pred_lasso_standard)

print("Dataset: Train")
print(f"MAE {train_mae_lasso_standard:.4f} \\ RMSE {train_rmse_lasso_standard:.4f} \\ R2 {train_r2_lasso_standard:.4f}")
print("Dataset: Test")
print(f"MAE {test_mae_lasso_standard:.4f} \\ RMSE {test_rmse_lasso_standard:.4f} \\ R2 {test_r2_lasso_standard:.4f}")

# Standard ElasticNet
print("\nStatus: Training ElasticNet (standard)")
elastic_standard_custom = ElasticNetRegressionSGD(learning_rate=learning_rate, n_iterations=n_iterations, alpha=0.1, l1_ratio=0.5)
elastic_standard_custom.fit(X_train_standard, y_train)

y_train_pred_elastic_standard = elastic_standard_custom.predict(X_train_standard)
y_test_pred_elastic_standard = elastic_standard_custom.predict(X_test_standard)

train_mae_elastic_standard = mean_absolute_error(y_train, y_train_pred_elastic_standard)
train_rmse_elastic_standard = np.sqrt(mean_squared_error(y_train, y_train_pred_elastic_standard))
train_r2_elastic_standard = r2_coefficient(y_train, y_train_pred_elastic_standard)

test_mae_elastic_standard = mean_absolute_error(y_test, y_test_pred_elastic_standard)
test_rmse_elastic_standard = np.sqrt(mean_squared_error(y_test, y_test_pred_elastic_standard))
test_r2_elastic_standard = r2_coefficient(y_test, y_test_pred_elastic_standard)

print("Dataset: Train")
print(f"MAE {train_mae_elastic_standard:.4f} \\ RMSE {train_rmse_elastic_standard:.4f} \\ R2 {train_r2_elastic_standard:.4f}")
print("Dataset: Test")
print(f"MAE {test_mae_elastic_standard:.4f} \\ RMSE {test_rmse_elastic_standard:.4f} \\ R2 {test_r2_elastic_standard:.4f}")

TRAINING MODELS WITH STANDARD NORMALIZATION

Status: Training Linear Regression (standard)
Dataset: Train
MAE 738.1556 \ RMSE 1059.9369 \ R2 0.5599
Dataset: Test
MAE 742.5138 \ RMSE 1222.6110 \ R2 0.4088

Status: Training Ridge (standard)
Dataset: Train
MAE 818.5574 \ RMSE 1177.1305 \ R2 0.4571
Dataset: Test
MAE 822.9199 \ RMSE 1238.1150 \ R2 0.3937

Status: Training Lasso (standard)
Dataset: Train
MAE 738.1453 \ RMSE 1059.9397 \ R2 0.5598
Dataset: Test
MAE 742.5036 \ RMSE 1222.5849 \ R2 0.4088

Status: Training ElasticNet (standard)
Dataset: Train
MAE 739.5308 \ RMSE 1063.6490 \ R2 0.5568
Dataset: Test
MAE 743.7758 \ RMSE 1215.3149 \ R2 0.4158


In [53]:
# 7.3 Adding the results to the dataframe with metrics on samples
print("UPDATING RESULTS TABLES WITH NORMALIZED DATA")

# Adding MinMax normalized results
minmax_models = [
    ['LinearRegression_MinMax', train_mae_lr_minmax, test_mae_lr_minmax],
    ['Ridge_MinMax', train_mae_ridge_minmax, test_mae_ridge_minmax],
    ['Lasso_MinMax', train_mae_lasso_minmax, test_mae_lasso_minmax],
    ['ElasticNet_MinMax', train_mae_elastic_minmax, test_mae_elastic_minmax]
]

for row in minmax_models:
    results_mae.loc[len(results_mae)] = row

minmax_rmse = [
    ['LinearRegression_MinMax', train_rmse_lr_minmax, test_rmse_lr_minmax],
    ['Ridge_MinMax', train_rmse_ridge_minmax, test_rmse_ridge_minmax],
    ['Lasso_MinMax', train_rmse_lasso_minmax, test_rmse_lasso_minmax],
    ['ElasticNet_MinMax', train_rmse_elastic_minmax, test_rmse_elastic_minmax]
]

for row in minmax_rmse:
    results_rmse.loc[len(results_rmse)] = row

minmax_r2 = [
    ['LinearRegression_MinMax', train_r2_lr_minmax, test_r2_lr_minmax],
    ['Ridge_MinMax', train_r2_ridge_minmax, test_r2_ridge_minmax],
    ['Lasso_MinMax', train_r2_lasso_minmax, test_r2_lasso_minmax],
    ['ElasticNet_MinMax', train_r2_elastic_minmax, test_r2_elastic_minmax]
]

for row in minmax_r2:
    results_r2.loc[len(results_r2)] = row

# Adding Standard normalized results
standard_models = [
    ['LinearRegression_Standard', train_mae_lr_standard, test_mae_lr_standard],
    ['Ridge_Standard', train_mae_ridge_standard, test_mae_ridge_standard],
    ['Lasso_Standard', train_mae_lasso_standard, test_mae_lasso_standard],
    ['ElasticNet_Standard', train_mae_elastic_standard, test_mae_elastic_standard]
]

for row in standard_models:
    results_mae.loc[len(results_mae)] = row

standard_rmse = [
    ['LinearRegression_Standard', train_rmse_lr_standard, test_rmse_lr_standard],
    ['Ridge_Standard', train_rmse_ridge_standard, test_rmse_ridge_standard],
    ['Lasso_Standard', train_rmse_lasso_standard, test_rmse_lasso_standard],
    ['ElasticNet_Standard', train_rmse_elastic_standard, test_rmse_elastic_standard]
]

for row in standard_rmse:
    results_rmse.loc[len(results_rmse)] = row

standard_r2 = [
    ['LinearRegression_Standard', train_r2_lr_standard, test_r2_lr_standard],
    ['Ridge_Standard', train_r2_ridge_standard, test_r2_ridge_standard],
    ['Lasso_Standard', train_r2_lasso_standard, test_r2_lasso_standard],
    ['ElasticNet_Standard', train_r2_elastic_standard, test_r2_elastic_standard]
]

for row in standard_r2:
    results_r2.loc[len(results_r2)] = row

print("\nFinal MAE Results Table:")
print(results_mae.to_string(index=False))
print("\nFinal RMSE Results Table:")
print(results_rmse.to_string(index=False))
print("\nFinal R2 Results Table:")
print(results_r2.to_string(index=False))

print("\nAll models trained with:")
print("- Original features (no normalization)")
print("- MinMax normalized features")
print("- Standard normalized features")
print("\nTotal models in results: ", len(results_mae))

UPDATING RESULTS TABLES WITH NORMALIZED DATA

Final MAE Results Table:
                    model       train        test
  LinearRegression_Custom  714.484532  718.512561
 LinearRegression_Sklearn  711.791166  716.845780
             Ridge_Custom  899.646690  901.948541
            Ridge_Sklearn  711.787958  716.841873
             Lasso_Custom  714.472530  718.500932
            Lasso_Sklearn  711.730426  716.781148
        ElasticNet_Custom  724.365273  727.728814
       ElasticNet_Sklearn  716.586109  720.589267
             Ridge_Custom  899.646690  901.948541
            Ridge_Sklearn  711.787958  716.841873
             Lasso_Custom  714.472530  718.500932
            Lasso_Sklearn  711.730426  716.781148
        ElasticNet_Custom  724.365273  727.728814
       ElasticNet_Sklearn  716.586109  720.589267
  LinearRegression_MinMax  710.320497  714.700305
             Ridge_MinMax 1074.095283 1072.665702
             Lasso_MinMax  710.286342  714.654301
        ElasticNet_MinMax  91

## 8. Overfit models

In [57]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd

# Creating polynomial features
print("Creating polynomial features, degree of 10")
print("Comment: 'interest_level' feature is present in the train data set only, for that reason only features 'bathrooms' and 'bedrooms' being used")
basic_features = ['bathrooms', 'bedrooms']

print(f"Basic features used: {basic_features}")
print("Status: Creating polynomial features of degree 10")

# Extracting basic features from thr training and test data
X_train_basic = cleaned_df_train[basic_features].values
X_test_basic = cleaned_df_test[basic_features].values

print(f"\nOriginal feature shape: {X_train_basic.shape}")

# Normalizing before creating polynomial features to prevent overflow
print("\nStatus: Normalizing features before polynomial transformation.")
scaler = StandardScaler()
X_train_basic_scaled = scaler.fit_transform(X_train_basic)
X_test_basic_scaled = scaler.transform(X_test_basic)
print("Status: Features normalized.")

Creating polynomial features, degree of 10
Comment: 'interest_level' feature is present in the train data set only, for that reason only features 'bathrooms' and 'bedrooms' being used
Basic features used: ['bathrooms', 'bedrooms']
Status: Creating polynomial features of degree 10

Original feature shape: (48379, 2)

Status: Normalizing features before polynomial transformation.
Status: Features normalized.


In [71]:
# Creating degree-10 polynomials
poly_10 = PolynomialFeatures(degree=10, include_bias=False)
X_train_poly_10 = poly_10.fit_transform(X_train_basic_scaled)
X_test_poly_10 = poly_10.transform(X_test_basic_scaled)

print(f"Polynomial features created: {X_train_poly_10.shape[1]} features")
print(f"Original features: {X_train_basic.shape[1]}")
print(f"Feature expansion: {X_train_basic.shape[1]} to {X_train_poly_10.shape[1]}")

# Training a Linear Regression
print("Status: training Linear Regression with degree-10 polynomials.")

lr_poly_10 = LinearRegression()
lr_poly_10.fit(X_train_poly_10, y_train)

y_train_pred_10 = lr_poly_10.predict(X_train_poly_10)
y_test_pred_10 = lr_poly_10.predict(X_test_poly_10)

# Metrics
train_mae_10 = mean_absolute_error(y_train, y_train_pred_10)
train_rmse_10 = np.sqrt(mean_squared_error(y_train, y_train_pred_10))
train_r2_10 = r2_score(y_train, y_train_pred_10)

test_mae_10 = mean_absolute_error(y_test, y_test_pred_10)
test_rmse_10 = np.sqrt(mean_squared_error(y_test, y_test_pred_10))
test_r2_10 = r2_score(y_test, y_test_pred_10)

print("\nDegree-10 Polynomial Results:")
print("Dataset: Train")
print(f"MAE {train_mae_10:,.2f} \\ RMSE {train_rmse_10:,.2f} \\ R2 {train_r2_10:.4f}")
print("Dataset: Test")
print(f"MAE {test_mae_10:,.2f} \\ RMSE {test_rmse_10:,.2f} \\ R2 {test_r2_10:.4f}")

# Analysis
print("\nRESULTS ANALYSIS ")
print(f"Train-test gap: {abs(train_r2_10 - test_r2_10):,.2f}")

if test_r2_10 < -1000:
    print("Status: FAILURE DETECTED")
    print(f"Train R2 = {train_r2_10:.4f}")
    print(f"Test R2 = {test_r2_10:,.0f}")
    print("Overfitting detected. The model memorized training noise (overfit). Polynomials make values grow exponentially outside training range, so the data gets misrepredented.")
    
    print(f"\nBase features (bathrooms, bedrooms): {X_train_basic.shape[1]}")
    print(f"Polynomial combinations created: {X_train_poly_10.shape[1]} ")

elif test_r2_10 < 0:
    print("\nStatus: SEVERE OVERFITTING DETECTED")
    print("Model performs worse than baseline (baseline predicting mean)")
    
else:
    print("\nStatus: Failure not detected.")

# Comparing to baseline
baseline_r2 = 0.0  ## mean baseline by def.
print("Performance:")
print(f"Mean baseline R2: {baseline_r2:.4f}")
print(f"Degree-10 polynomial R2: {test_r2_10:.4f}")
print(f"Difference: {test_r2_10 - baseline_r2:,.2f}")

# Predictions
print("\nTraining predictions:")
print(f"Min: {y_train_pred_10.min():,.2f}")
print(f"Max: {y_train_pred_10.max():,.2f}")
print(f"Mean: {y_train_pred_10.mean():,.2f}")

print("\nTesting predictions:")
print(f"Min: {y_test_pred_10.min():,.2f}")
print(f"Max: {y_test_pred_10.max():,.2f}")
print(f"Mean: {y_test_pred_10.mean():,.2f}")

print("\nActual test values:")
print(f"Min: {y_test.min():,.2f}")
print(f"Max: {y_test.max():,.2f}")
print(f"Mean: {y_test.mean():,.2f}")

if abs(y_test_pred_10.mean()) > 1e6:
    print("\nAttention: Predictions appear too large. Sign of failure")

# Comment
print("\nComment: Degree-10 polynomials demonstrate overfitting")
print("For that reason, degree-3 polynomials used instead (see the next cell)")

Polynomial features created: 65 features
Original features: 2
Feature expansion: 2 to 65
Status: training Linear Regression with degree-10 polynomials.

Degree-10 Polynomial Results:
Dataset: Train
MAE 758.48 \ RMSE 1,081.91 \ R2 0.5414
Dataset: Test
MAE 53,484,552,954,600,208.00 \ RMSE 14,472,093,524,461,660,160.00 \ R2 -82836045503354039945354851385344.0000

RESULTS ANALYSIS 
Train-test gap: 82,836,045,503,354,039,945,354,851,385,344.00
Status: FAILURE DETECTED
Train R2 = 0.5414
Test R2 = -82,836,045,503,354,039,945,354,851,385,344
Overfitting detected. The model memorized training noise (overfit). Polynomials make values grow exponentially outside training range, so the data gets misrepredented.

Base features (bathrooms, bedrooms): 2
Polynomial combinations created: 65 
Performance:
Mean baseline R2: 0.0000
Degree-10 polynomial R2: -82836045503354039945354851385344.0000
Difference: -82,836,045,503,354,039,945,354,851,385,344.00

Training predictions:
Min: 2,046.93
Max: 11,678.35
Me

In [72]:
# Creating polynomial features on scaled data
poly = PolynomialFeatures(degree=3, include_bias=False)  ## reduced from 10 to 3
X_train_poly = poly.fit_transform(X_train_basic_scaled)
X_test_poly = poly.transform(X_test_basic_scaled)

print(f"Polynomial feature shape: {X_train_poly.shape}")
print(f"Number of polynomial features created: {X_train_poly.shape[1]}")
print("\nReminder for further use. Created combinations of the kind:")
print("- bathrooms, bedrooms, interest_level (original)")
print("- bathrooms^2, bedrooms^2, bathrooms*bedrooms, etc.")
print("- bathrooms^3, bathrooms^2*bedrooms, etc.")
print("So on up to degree specified.")

Polynomial feature shape: (48379, 9)
Number of polynomial features created: 9

Reminder for further use. Created combinations of the kind:
- bathrooms, bedrooms, interest_level (original)
- bathrooms^2, bedrooms^2, bathrooms*bedrooms, etc.
- bathrooms^3, bathrooms^2*bedrooms, etc.
So on up to degree specified.


In [73]:
# 8.3 Training all models on polynomial features
print("TRAINING MODELS ON POLYNOMIAL FEATURES")
print("Comment: sklearn models used for polynomial features due to numerical stability (custom SGD being unstable with high-degree polynomials)")

# Linear Regression (polynomial). Sklearn
print("Status: Training Linear Regression (poly)")
lr_poly_sklearn = LinearRegression()
lr_poly_sklearn.fit(X_train_poly, y_train)

y_train_pred_lr_poly = lr_poly_sklearn.predict(X_train_poly)
y_test_pred_lr_poly = lr_poly_sklearn.predict(X_test_poly)

train_mae_lr_poly = mean_absolute_error(y_train, y_train_pred_lr_poly)
train_rmse_lr_poly = np.sqrt(mean_squared_error(y_train, y_train_pred_lr_poly))
train_r2_lr_poly = r2_score(y_train, y_train_pred_lr_poly)

test_mae_lr_poly = mean_absolute_error(y_test, y_test_pred_lr_poly)
test_rmse_lr_poly = np.sqrt(mean_squared_error(y_test, y_test_pred_lr_poly))
test_r2_lr_poly = r2_score(y_test, y_test_pred_lr_poly)

print("Dataset: Train")
print(f"MAE {train_mae_lr_poly:.4f} \\ RMSE {train_rmse_lr_poly:.4f} \\ R2 {train_r2_lr_poly:.4f}")
print("Dataset: Test")
print(f"MAE {test_mae_lr_poly:.4f} \\ RMSE {test_rmse_lr_poly:.4f} \\ R2 {test_r2_lr_poly:.4f}")

# Ridge (polynomial). Sklearn
print("\nStatus: Training Ridge (poly), alpha=1.0")
ridge_poly_sklearn = Ridge(alpha=1.0, random_state=21)
ridge_poly_sklearn.fit(X_train_poly, y_train)

y_train_pred_ridge_poly = ridge_poly_sklearn.predict(X_train_poly)
y_test_pred_ridge_poly = ridge_poly_sklearn.predict(X_test_poly)

train_mae_ridge_poly = mean_absolute_error(y_train, y_train_pred_ridge_poly)
train_rmse_ridge_poly = np.sqrt(mean_squared_error(y_train, y_train_pred_ridge_poly))
train_r2_ridge_poly = r2_score(y_train, y_train_pred_ridge_poly)

test_mae_ridge_poly = mean_absolute_error(y_test, y_test_pred_ridge_poly)
test_rmse_ridge_poly = np.sqrt(mean_squared_error(y_test, y_test_pred_ridge_poly))
test_r2_ridge_poly = r2_score(y_test, y_test_pred_ridge_poly)

print("Dataset: Train")
print(f"MAE {train_mae_ridge_poly:.4f} \\ RMSE {train_rmse_ridge_poly:.4f} \\ R2 {train_r2_ridge_poly:.4f}")
print("Dataset: Test")
print(f"MAE {test_mae_ridge_poly:.4f} \\ RMSE {test_rmse_ridge_poly:.4f} \\ R2 {test_r2_ridge_poly:.4f}")

# Lasso (polynomial). Sklearn
print("\nStatus: Training Lasso (poly), alpha=0.1")
lasso_poly_sklearn = Lasso(alpha=0.1, random_state=21, max_iter=5000)
lasso_poly_sklearn.fit(X_train_poly, y_train)

y_train_pred_lasso_poly = lasso_poly_sklearn.predict(X_train_poly)
y_test_pred_lasso_poly = lasso_poly_sklearn.predict(X_test_poly)

train_mae_lasso_poly = mean_absolute_error(y_train, y_train_pred_lasso_poly)
train_rmse_lasso_poly = np.sqrt(mean_squared_error(y_train, y_train_pred_lasso_poly))
train_r2_lasso_poly = r2_score(y_train, y_train_pred_lasso_poly)

test_mae_lasso_poly = mean_absolute_error(y_test, y_test_pred_lasso_poly)
test_rmse_lasso_poly = np.sqrt(mean_squared_error(y_test, y_test_pred_lasso_poly))
test_r2_lasso_poly = r2_score(y_test, y_test_pred_lasso_poly)

print("Dataset: Train")
print(f"MAE {train_mae_lasso_poly:.4f} \\ RMSE {train_rmse_lasso_poly:.4f} \\ R2 {train_r2_lasso_poly:.4f}")
print("Dataset: Test")
print(f"MAE {test_mae_lasso_poly:.4f} \\ RMSE {test_rmse_lasso_poly:.4f} \\ R2 {test_r2_lasso_poly:.4f}")

# ElasticNet (polynomial). Sklearn
print("\nStatus: Training ElasticNet (poly) with alpha=0.1")
elastic_poly_sklearn = ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=21, max_iter=5000)
elastic_poly_sklearn.fit(X_train_poly, y_train)

y_train_pred_elastic_poly = elastic_poly_sklearn.predict(X_train_poly)
y_test_pred_elastic_poly = elastic_poly_sklearn.predict(X_test_poly)

train_mae_elastic_poly = mean_absolute_error(y_train, y_train_pred_elastic_poly)
train_rmse_elastic_poly = np.sqrt(mean_squared_error(y_train, y_train_pred_elastic_poly))
train_r2_elastic_poly = r2_score(y_train, y_train_pred_elastic_poly)

test_mae_elastic_poly = mean_absolute_error(y_test, y_test_pred_elastic_poly)
test_rmse_elastic_poly = np.sqrt(mean_squared_error(y_test, y_test_pred_elastic_poly))
test_r2_elastic_poly = r2_score(y_test, y_test_pred_elastic_poly)

print("Dataset: Train")
print(f"MAE {train_mae_elastic_poly:.4f} \\ RMSE {train_rmse_elastic_poly:.4f} \\ R2 {train_r2_elastic_poly:.4f}")
print("Dataset: Test")
print(f"MAE {test_mae_elastic_poly:.4f} \\ RMSE {test_rmse_elastic_poly:.4f} \\ R2 {test_r2_elastic_poly:.4f}")

TRAINING MODELS ON POLYNOMIAL FEATURES
Comment: sklearn models used for polynomial features due to numerical stability (custom SGD being unstable with high-degree polynomials)
Status: Training Linear Regression (poly)
Dataset: Train
MAE 763.5438 \ RMSE 1093.8162 \ R2 0.5313
Dataset: Test
MAE 3038.2076 \ RMSE 611411.9507 \ R2 -147850.0740

Status: Training Ridge (poly), alpha=1.0
Dataset: Train
MAE 763.5441 \ RMSE 1093.8162 \ R2 0.5313
Dataset: Test
MAE 3038.2691 \ RMSE 611428.4039 \ R2 -147858.0315

Status: Training Lasso (poly), alpha=0.1
Dataset: Train
MAE 763.5380 \ RMSE 1093.8163 \ R2 0.5313
Dataset: Test
MAE 3039.9499 \ RMSE 611882.3341 \ R2 -148077.6568

Status: Training ElasticNet (poly) with alpha=0.1
Dataset: Train
MAE 764.9696 \ RMSE 1094.9551 \ R2 0.5303
Dataset: Test
MAE 3161.1114 \ RMSE 644101.1095 \ R2 -164082.4322


In [74]:
# 8.4 Storing the results of the quality metrics
print("RESULTS TABLES UPD")

poly_models_mae = [
    ['LinearRegression_Poly', train_mae_lr_poly, test_mae_lr_poly],
    ['Ridge_Poly_alpha1.0', train_mae_ridge_poly, test_mae_ridge_poly],
    ['Lasso_Poly_alpha0.1', train_mae_lasso_poly, test_mae_lasso_poly],
    ['ElasticNet_Poly_alpha0.1', train_mae_elastic_poly, test_mae_elastic_poly]
]

for row in poly_models_mae:
    results_mae.loc[len(results_mae)] = row

poly_models_rmse = [
    ['LinearRegression_Poly', train_rmse_lr_poly, test_rmse_lr_poly],
    ['Ridge_Poly_alpha1.0', train_rmse_ridge_poly, test_rmse_ridge_poly],
    ['Lasso_Poly_alpha0.1', train_rmse_lasso_poly, test_rmse_lasso_poly],
    ['ElasticNet_Poly_alpha0.1', train_rmse_elastic_poly, test_rmse_elastic_poly]
]

for row in poly_models_rmse:
    results_rmse.loc[len(results_rmse)] = row

poly_models_r2 = [
    ['LinearRegression_Poly', train_r2_lr_poly, test_r2_lr_poly],
    ['Ridge_Poly_alpha1.0', train_r2_ridge_poly, test_r2_ridge_poly],
    ['Lasso_Poly_alpha0.1', train_r2_lasso_poly, test_r2_lasso_poly],
    ['ElasticNet_Poly_alpha0.1', train_r2_elastic_poly, test_r2_elastic_poly]
]

for row in poly_models_r2:
    results_r2.loc[len(results_r2)] = row

print("Status: Results tables updated")

RESULTS TABLES UPD
Status: Results tables updated


In [83]:
# 8.5 Analyzing results
print("RESULTS ANALYZIS")
print("\nTrain-Test performance gaps (signs of overfitting):")
print(f"Linear Regression (poly): train R2={train_r2_lr_poly:.4f} \\ test R2={test_r2_lr_poly:.4f} \\ gap={train_r2_lr_poly - test_r2_lr_poly:.4f}")
print(f"Ridge (poly):            train R2={train_r2_ridge_poly:.4f} \\ test R2={test_r2_ridge_poly:.4f} \\ gap={train_r2_ridge_poly - test_r2_ridge_poly:.4f}")
print(f"Lasso (poly):            train R2={train_r2_lasso_poly:.4f} \\ test R2={test_r2_lasso_poly:.4f} \\ gap={train_r2_lasso_poly - test_r2_lasso_poly:.4f}")
print(f"ElasticNet (poly):       train R2={train_r2_elastic_poly:.4f} \\ test R2={test_r2_elastic_poly:.4f} \\ gap={train_r2_elastic_poly - test_r2_elastic_poly:.4f}")

print("\nResults interpretation")
print("Large gap (>0.1): model is overfitting (memorizing training data)")
print("Small gap (<0.05): model generalizes well")
print("Regularization - Ridge/Lasso/ElasticNet - should reduce overfitting")

# 8.6 Trying different alpha values
print("\nTESTING DIFFERENT ALPHA VALUES")
alphas_to_test = [0.01, 0.1, 1.0, 10.0, 100.0]

print("Status: Testing Ridge with different alphas")
ridge_results = []

for alpha in alphas_to_test:
    ridge_test = Ridge(alpha=alpha, random_state=21)
    ridge_test.fit(X_train_poly, y_train)
    
    train_pred = ridge_test.predict(X_train_poly)
    test_pred = ridge_test.predict(X_test_poly)
    
    train_r2 = r2_score(y_train, train_pred)
    test_r2 = r2_score(y_test, test_pred)
    gap = train_r2 - test_r2
    
    ridge_results.append([f'Ridge_Poly_alpha{alpha}', train_r2, test_r2, gap])
    print(f"alpha={alpha:6.2f}: Train R2={train_r2:.4f} \\ Test R2={test_r2:.4f} \\ Gap={gap:.4f}")

print("\nBest Ridge alpha based on test R2:")
best_ridge = max(ridge_results, key=lambda x: x[2])  ## max test R2
print(f"{best_ridge[0]}: Test R2={best_ridge[2]:.4f}, Gap={best_ridge[3]:.4f}")

print("\nStatus: Testing Lasso with different alphas")
lasso_results = []

for alpha in alphas_to_test:
    lasso_test = Lasso(alpha=alpha, random_state=21, max_iter=5000)
    lasso_test.fit(X_train_poly, y_train)
    
    train_pred = lasso_test.predict(X_train_poly)
    test_pred = lasso_test.predict(X_test_poly)
    
    train_r2 = r2_score(y_train, train_pred)
    test_r2 = r2_score(y_test, test_pred)
    gap = train_r2 - test_r2
    
    lasso_results.append([f'Lasso_Poly_alpha{alpha}', train_r2, test_r2, gap])
    print(f"alpha={alpha:6.2f} \\ train R2={train_r2:.4f} \\ test R2={test_r2:.4f} \\ gap={gap:.4f}")

print("\nBest Lasso alpha based on test R2:")
best_lasso = max(lasso_results, key=lambda x: x[2])
print(f"{best_lasso[0]}: Test R2={best_lasso[2]:.4f}, Gap={best_lasso[3]:.4f}")

print("\nNote: Alpha parameter controls regularization strength")
print("Higher alpha = more regularization = less overfitting (but may underfit)")
print("Lower alpha = less regularization = better fit (but may overfit)")

RESULTS ANALYZIS

Train-Test performance gaps (signs of overfitting):
Linear Regression (poly): train R2=0.5313 \ test R2=-147850.0740 \ gap=147850.6053
Ridge (poly):            train R2=0.5313 \ test R2=-147858.0315 \ gap=147858.5627
Lasso (poly):            train R2=0.5313 \ test R2=-148077.6568 \ gap=148078.1881
ElasticNet (poly):       train R2=0.5303 \ test R2=-164082.4322 \ gap=164082.9624

Results interpretation
Large gap (>0.1): model is overfitting (memorizing training data)
Small gap (<0.05): model generalizes well
Regularization - Ridge/Lasso/ElasticNet - should reduce overfitting

TESTING DIFFERENT ALPHA VALUES
Status: Testing Ridge with different alphas
alpha=  0.01: Train R2=0.5313 \ Test R2=-147850.1536 \ Gap=147850.6848
alpha=  0.10: Train R2=0.5313 \ Test R2=-147850.8698 \ Gap=147851.4011
alpha=  1.00: Train R2=0.5313 \ Test R2=-147858.0315 \ Gap=147858.5627
alpha= 10.00: Train R2=0.5313 \ Test R2=-147929.5966 \ Gap=147930.1279
alpha=100.00: Train R2=0.5313 \ Test R2=-

## 9. Naive models

In [93]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd

# 9.1 Calculating the mean and the median baselines
print("CALCULATING MEAN AND MEDIAN")

# Mean baseline, predicting the mean of training prices for all samples
mean_price = np.mean(y_train)
print(f"Mean price from training data: {mean_price:,.2f}")

# Median baseline, predicting the median of training prices for all samples
median_price = np.median(y_train)
print(f"Median price from training data: {median_price:,.2f}")

# Creating predictions (same value for all samples)
y_train_pred_mean = np.full(len(y_train), mean_price)
y_test_pred_mean = np.full(len(y_test), mean_price)

y_train_pred_median = np.full(len(y_train), median_price)
y_test_pred_median = np.full(len(y_test), median_price)

# Calculating metrics for Mean baseline
train_mae_mean = mean_absolute_error(y_train, y_train_pred_mean)
train_rmse_mean = np.sqrt(mean_squared_error(y_train, y_train_pred_mean))
train_r2_mean = r2_score(y_train, y_train_pred_mean)

test_mae_mean = mean_absolute_error(y_test, y_test_pred_mean)
test_rmse_mean = np.sqrt(mean_squared_error(y_test, y_test_pred_mean))
test_r2_mean = r2_score(y_test, y_test_pred_mean)

print("\nDataset: Train")
print(f"MAE {train_mae_mean:.4f} \\ RMSE  {train_rmse_mean:.4f}  \\ R2 {train_r2_mean:.4f}")
print("Dataset: Test")
print(f"MAE {test_mae_mean:.4f}  \\ RMSE {test_rmse_mean:.4f} \\ R2 {test_r2_mean:.4f}")

print("Comment: R2 = 0 for mean baseline (by definition)")

# Calculating metrics for Median baseline
train_mae_median = mean_absolute_error(y_train, y_train_pred_median)
train_rmse_median = np.sqrt(mean_squared_error(y_train, y_train_pred_median))
train_r2_median = r2_score(y_train, y_train_pred_median)

test_mae_median = mean_absolute_error(y_test, y_test_pred_median)
test_rmse_median = np.sqrt(mean_squared_error(y_test, y_test_pred_median))
test_r2_median = r2_score(y_test, y_test_pred_median)

print("\nDataset: Train")
print(f"MAE {train_mae_median:.4f} \\ RMSE {train_rmse_median:.4f} \\ R2 {train_r2_median:.4f}")
print("Dataset: Test")
print(f"MAE {test_mae_median:.4f} \\ RMSE {test_rmse_median:.4f} \\ R2 {test_r2_median:.4f}")

# Adding to results dataframes
print("\nUPDATING THE RESULTS TABLES")

# Adding Mean baseline
results_mae.loc[len(results_mae)] = ['Mean_Baseline', train_mae_mean, test_mae_mean]
results_rmse.loc[len(results_rmse)] = ['Mean_Baseline', train_rmse_mean, test_rmse_mean]
results_r2.loc[len(results_r2)] = ['Mean_Baseline', train_r2_mean, test_r2_mean]

# Adding Median baseline
results_mae.loc[len(results_mae)] = ['Median_Baseline', train_mae_median, test_mae_median]
results_rmse.loc[len(results_rmse)] = ['Median_Baseline', train_rmse_median, test_rmse_median]
results_r2.loc[len(results_r2)] = ['Median_Baseline', train_r2_median, test_r2_median]

print("Status: results tables updated. Baseline models added")

# Comparing with best performing models so far
print("\nMAE Comparison (test set, lower == better):")
print(f"Mean Baseline: {test_mae_mean:.4f}")
print(f"Median Baseline: {test_mae_median:.4f}")
print(f"Best previous model: {results_mae['test'].min():.4f} (excl. polynomials)")

print("\nR2 Comparison (test set, higher == better):")
print(f"Mean Baseline: {test_r2_mean:.4f}")
print(f"Median Baseline: {test_r2_median:.4f}")
# Geting best R2 that's not from polynomial models
valid_r2 = results_r2[~results_r2['model'].str.contains('Poly', na=False)]
best_r2 = valid_r2['test'].max()
print(f"Best previous model: {best_r2:.4f}")

CALCULATING MEAN AND MEDIAN
Mean price from training data: 3,538.64
Median price from training data: 3,150.00

Dataset: Train
MAE 1139.1925 \ RMSE  1597.6467  \ R2 0.0000
Dataset: Test
MAE 1137.3399  \ RMSE 1590.1030 \ R2 -0.0000
Comment: R2 = 0 for mean baseline (by definition)

Dataset: Train
MAE 1086.2105 \ RMSE 1644.2364 \ R2 -0.0592
Dataset: Test
MAE 1084.2746 \ RMSE 1635.3760 \ R2 -0.0578

UPDATING THE RESULTS TABLES
Status: results tables updated. Baseline models added

MAE Comparison (test set, lower == better):
Mean Baseline: 1137.3399
Median Baseline: 1084.2746
Best previous model: 714.6543 (excl. polynomials)

R2 Comparison (test set, higher == better):
Mean Baseline: -0.0000
Median Baseline: -0.0578
Best previous model: 0.4694


## 10. Comparing results

In [101]:
import pandas as pd
import numpy as np

# 10.1 Printing final tables (excl. polynomials)
print("FIN RESULTS TABLES")

# Filtering out polynomial models (failed)
valid_models_mask = ~results_mae['model'].str.contains('Poly', na=False)

# Cleaning tables (removing duplicates if applicable)
results_mae_clean = results_mae[valid_models_mask].drop_duplicates(subset=['model']).reset_index(drop=True)
results_rmse_clean = results_rmse[valid_models_mask].drop_duplicates(subset=['model']).reset_index(drop=True)
results_r2_clean = results_r2[valid_models_mask].drop_duplicates(subset=['model']).reset_index(drop=True)

print("\nMAE, Mean Absolute Error")
print(results_mae_clean.sort_values('test').to_string(index=False))

print("\nRMSE, Root Mean Squared Error")
print(results_rmse_clean.sort_values('test').to_string(index=False))

print("\nR2, R-Squared")
print(results_r2_clean.sort_values('test', ascending=False).to_string(index=False))

# 10.2 Finding the best model
print("\nTHE BEST MODEL")

# Finding best models by different metrics
best_mae_idx = results_mae_clean['test'].idxmin()
best_mae_model = results_mae_clean.loc[best_mae_idx]

best_rmse_idx = results_rmse_clean['test'].idxmin()
best_rmse_model = results_rmse_clean.loc[best_rmse_idx]

best_r2_idx = results_r2_clean['test'].idxmax()
best_r2_model = results_r2_clean.loc[best_r2_idx]

print("\nBest model by Test MAE")
print(f"Model: {best_mae_model['model']}")
print(f"Test MAE: {best_mae_model['test']:.4f}")
print(f"Train MAE: {best_mae_model['train']:.4f}")

print("\nBest model by Test RMSE")
print(f"Model: {best_rmse_model['model']}")
print(f"Test RMSE: {best_rmse_model['test']:.4f}")
print(f"Train RMSE: {best_rmse_model['train']:.4f}")

print("\nBest model by Test R2")
print(f"Model: {best_r2_model['model']}")
print(f"Test R2: {best_r2_model['test']:.4f}")
print(f"Train R2: {best_r2_model['train']:.4f}")

# Finding overall best model
## Counting how many times each model appears as "best"
best_models = [best_mae_model['model'], best_rmse_model['model'], best_r2_model['model']]
from collections import Counter
best_count = Counter(best_models)
overall_best = best_count.most_common(1)[0][0]

print(f"\nOverall best model: {overall_best}")
print(f"Appears as 'best' across {best_count[overall_best]} out of 3 metrics.")

# Getting all metrics for the best model
best_mae_val = results_mae_clean[results_mae_clean['model'] == overall_best].iloc[0]
best_rmse_val = results_rmse_clean[results_rmse_clean['model'] == overall_best].iloc[0]
best_r2_val = results_r2_clean[results_r2_clean['model'] == overall_best].iloc[0]

print(f"\nPerformance Summary:")
print(f"Test MAE:  {best_mae_val['test']:.2f} average error per prediction")
print(f"Test RMSE: {best_rmse_val['test']:.2f}")
print(f"Test R2:   {best_r2_val['test']:.4f} (explains {best_r2_val['test']*100:.2f}% of variance)")

FIN RESULTS TABLES

MAE, Mean Absolute Error
                    model       train        test
             Lasso_MinMax  710.286342  714.654301
  LinearRegression_MinMax  710.320497  714.700305
            Lasso_Sklearn  711.730426  716.781148
            Ridge_Sklearn  711.787958  716.841873
 LinearRegression_Sklearn  711.791166  716.845780
             Lasso_Custom  714.472530  718.500932
  LinearRegression_Custom  714.484532  718.512561
       ElasticNet_Sklearn  716.586109  720.589267
        ElasticNet_Custom  724.365273  727.728814
           Lasso_Standard  738.145338  742.503598
LinearRegression_Standard  738.155586  742.513840
      ElasticNet_Standard  739.530832  743.775813
           Ridge_Standard  818.557404  822.919856
             Ridge_Custom  899.646690  901.948541
        ElasticNet_MinMax  914.442407  913.722282
             Ridge_MinMax 1074.095283 1072.665702
          Median_Baseline 1086.210505 1084.274571
            Mean_Baseline 1139.192515 1137.339895

RMSE

In [107]:
# 10.3 Most stable model
print("THE MOST STABLE MODEL")
print("Comment: Stability is measured by the gap between train and test performance.")
print("Smaller gap == better generalization == more stable model")

# Calculating train-test gaps for all models
results_mae_clean['gap'] = abs(results_mae_clean['train'] - results_mae_clean['test'])
results_r2_clean['gap'] = abs(results_r2_clean['train'] - results_r2_clean['test'])

print("\nTrain-Test gap analysis (MAE)")
mae_stability = results_mae_clean[['model', 'train', 'test', 'gap']].sort_values('gap')
print(mae_stability.head(10).to_string(index=False))

print("\nTrain-Test Gap Analysis (R2)")
r2_stability = results_r2_clean[['model', 'train', 'test', 'gap']].sort_values('gap')
print(r2_stability.head(10).to_string(index=False))

# Most stable model
most_stable_mae = mae_stability.iloc[0]['model']
most_stable_r2 = r2_stability.iloc[0]['model']

print("\nMOST STABLE MODEL")
if most_stable_mae == most_stable_r2:
    print(f"The most stable model: {most_stable_mae}")
    print(f"\nThis model has the smallest train-test gap across both MAE and R2 metrics.")
else:
    print(f"\nThe most stable model by MAE gap: {most_stable_mae}")
    print(f"The most stable model by R2 gap:  {most_stable_r2}")

THE MOST STABLE MODEL
Comment: Stability is measured by the gap between train and test performance.
Smaller gap == better generalization == more stable model

Train-Test gap analysis (MAE)
                  model       train        test      gap
      ElasticNet_MinMax  914.442407  913.722282 0.720125
           Ridge_MinMax 1074.095283 1072.665702 1.429582
          Mean_Baseline 1139.192515 1137.339895 1.852620
        Median_Baseline 1086.210505 1084.274571 1.935933
           Ridge_Custom  899.646690  901.948541 2.301851
      ElasticNet_Custom  724.365273  727.728814 3.363541
     ElasticNet_Sklearn  716.586109  720.589267 4.003158
LinearRegression_Custom  714.484532  718.512561 4.028029
           Lasso_Custom  714.472530  718.500932 4.028402
    ElasticNet_Standard  739.530832  743.775813 4.244981

Train-Test Gap Analysis (R2)
              model     train      test      gap
      Mean_Baseline  0.000000 -0.000016 0.000016
       Ridge_MinMax  0.070628  0.071355 0.000727
  Elast